In [432]:
#Importacion de librerias
import requests
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
from unidecode import unidecode
import numpy as np
from tqdm import tqdm
import json
from pandas import json_normalize
import openpyxl
import concurrent.futures
from googleapiclient.discovery import build
from google.oauth2 import service_account


periodos = ['26ES5', '2026D', '26V05']
periodo_virtual = '26V05'
periodo_presencial = '2026D'
periodo_especializacion = '26ES5'
lider = 'SANDRA'
periodo_pago = '202609'

In [318]:
#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   #url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print('Error:',e)

Conexion a la base de datos realizada


In [319]:
# Autenticación con la clave de servicio
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly']
KEY = 'key2.json'  # LLAVES
SPREADSHEET_ID = "15NkOzO6MBlMJjtabMTB1Bc_PEKJ6lb6Rh8won1i1xm0"  # ID DE LA HOJA DE CALCULO

# Crear credenciales
creds = service_account.Credentials.from_service_account_file(KEY, scopes=SCOPES)
service = build('sheets', 'v4', credentials=creds)
sheet = service.spreadsheets()

# Llamar a la API de Google Sheets
RANGE_NAME = "'VENTAS'!A1:AE100000000"  # HOJA Y RANGO A TRABAJAR

result = sheet.values().get(spreadsheetId=SPREADSHEET_ID, range=RANGE_NAME).execute()
values = result.get('values', [])

# Mostrar resultados
if not values:
    print("No se encontraron datos.")
else:
    print(f"Datos obtenidos: {len(values):,} filas")

Datos obtenidos: 21,768 filas


In [320]:
df_excel = pd.DataFrame(values[1:], columns=values[0])
df_excel.shape

(21767, 31)

In [321]:
df_pre_ventas = df_excel[df_excel['Periodo'].isin(periodos)].copy()
df_pre_ventas

,Hora agregado,Dirección IP,Numero de recibo,Propietario del registro,Nombre de Estudiante,Ciudad de residencia,Número de Documento de estudiante,Tipo de inscripción,Programa vendido,Modalidad,...,% De Descuento,Valor pagado,Medio de pago,Fuente de pionero,Campana,Fuente campana,Content,Sitio web,ID único,Fecha
12553,05-16-2026 13:21:07,181.128.120.184,9738-1105460763-260514043236,Vinculaciones Margarita Rosa Navarro Garzon,capera varón Juan pablo,Colombia - Tolima - Ibague,1105460763,Normal,ADMINISTRACION PUBLICA,Virtual,...,60%,"$90.000,00",Pse,,,,,,13529,16/05/2026
13165,05-27-2026 10:38:12,190.184.201.12,9741-1118365041-260425115705,Vinculaciones Laura Yovana Bedoya Caballos,LAURA OLARTE LOPEZ,Colombia - Antioquia - Medellin,1118365041,Normal,ADMINISTRACION DE EMPRESAS,Presencial,...,60%,"$700.000,00",Banco,,,,,,14141,27/05/2026
15331,07-30-2026 12:43:33,191.156.159.249,9870-88242954-260722093908,Vinculaciones David Alejandro Orjuela Chaparro,HECTOR ALEXANDER HIGUERA DE LOS REYES,Colombia - Boyaca - Tunja,88242954,Normal,INGENIERIA INDUSTRIAL,Virtual,...,"0,1","$83.674,00",Pse,,,,,,16391,30/07/2026
15364,07-30-2026 15:00:02,38.210.96.76,9889-16499517-260728010742,Vinculaciones Miryam Daniela Callejas Perez,Carlos Hernán Valencia Hurtado,Colombia - Valle Del Cauca - Buenaventura,16499517,Normal,ESPECIALIZACIÓN EN GERENCIA EDUCATIVA,Virtual,...,"0,6","$193.948,00",Banco,,,,,,16424,30/07/2026
15468,07-31-2026 10:45:51,38.210.96.72,9895-1122141658-260729021731,Vinculaciones Miryam Daniela Callejas Perez,Luis Alberto Cordoba Gonzalez,Colombia - Meta - Villavicencio,1122141658,Normal,ESPECIALIZACION EN CONTRATACION ESTATAL,Virtual,...,"0,6","$193.948,00",Pse,,,,,,16528,31/07/2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20845,10-06-2026 22:05:36,191.156.18.44,10026-1044218402-261004060249,Vinculaciones Jesus David Jimenez Amado,RAFAEL EDUARDO ALVEAR MEDINA,Colombia - Atlantico - Barranquilla,1044218402,Normal,NEGOCIOS INTERNACIONALES,Virtual,...,"0,6","$42.400,00",Pse,,,,,,20870,6/10/2026
20846,10-06-2026 22:23:19,191.156.18.44,9604-1066098610-260930061607,Vinculaciones Jesus David Jimenez Amado,Ingrid Gissela Carrero Carrascal,Colombia - Cesar - Curumani,1066098610,Normal,ESPECIALIZACION EN CONTRATACION ESTATAL,Virtual,...,"0,6","$154.360,00",Pse,,,,,,20871,6/10/2026
21763,,,,,,,,,,,...,None,None,None,None,None,None,None,None,None,None
21764,,,,,,,,,,,...,None,None,None,None,None,None,None,None,None,None


# SE COMPLEMENTA LA INFORMACION DE LOS ASESORES

In [322]:
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'

try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    #archivo_orig = pd.read_sql("select * from CUN_REPOSITORIO.Comisiones.Com_Telec_Oct_2024_2_yCT", engine2_1)
    df_cedula = pd.read_sql(f'''
                    SELECT IDENTIFICACION AS 'Documento Ejecutivo', BOX_MAIL AS 'Correo Ejecutivo' FROM Hub_Empleados
                               ''', engine2_1)
    df_cedula_2 = df_cedula.copy()
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")
df_cedula_2.shape

Conexion a la base de datos realizada
Conexión cerrada


(89770, 2)

In [323]:
df_cedula_2['Documento Ejecutivo'] = df_cedula_2['Documento Ejecutivo'].astype(int).astype(str)
df_cedula_2['Correo Ejecutivo'] = df_cedula_2['Correo Ejecutivo'].str.upper()
df_cedula_2

,Documento Ejecutivo,Correo Ejecutivo
0,28538359,DIANA_ACOSTAC@CUN.EDU.CO
1,1075234239,YENNY_DIAZ@CUN.EDU.CO
2,36953386,ASTRID_ERASO@CUN.EDU.CO
3,19385060,GERMAN_IBAGON@CUN.EDU.CO
4,1072366328,JOHAN_RODRIGUEZ@CUN.EDU.CO
...,...,...
89765,1074128157,YINER_UMANAG@CUN.EDU.CO
89766,1110583837,DANNA_PEREZ@CUN.EDU.CO
89767,1121838723,YAMILE_RODRIGUEZ@CUN.EDU.CO
89768,1141318847,SARA_PASTRANAR@CUN.EDU.CO


In [324]:
df_pre_ventas['Correo Ejecutivo'] = df_pre_ventas['Correo Ejecutivo'].str.upper()

In [325]:
# se unifica con la informacion de registro ventas
df_pre_ventas = df_pre_ventas.merge(df_cedula_2,how='inner',left_on='Correo Ejecutivo',right_on='Correo Ejecutivo').copy()
df_pre_ventas.drop_duplicates(inplace=True)
df_pre_ventas

,Hora agregado,Dirección IP,Numero de recibo,Propietario del registro,Nombre de Estudiante,Ciudad de residencia,Número de Documento de estudiante,Tipo de inscripción,Programa vendido,Modalidad,...,Valor pagado,Medio de pago,Fuente de pionero,Campana,Fuente campana,Content,Sitio web,ID único,Fecha,Documento Ejecutivo_y
0,05-16-2026 13:21:07,181.128.120.184,9738-1105460763-260514043236,Vinculaciones Margarita Rosa Navarro Garzon,capera varón Juan pablo,Colombia - Tolima - Ibague,1105460763,Normal,ADMINISTRACION PUBLICA,Virtual,...,"$90.000,00",Pse,,,,,,13529,16/05/2026,1110587566
49,05-27-2026 10:38:12,190.184.201.12,9741-1118365041-260425115705,Vinculaciones Laura Yovana Bedoya Caballos,LAURA OLARTE LOPEZ,Colombia - Antioquia - Medellin,1118365041,Normal,ADMINISTRACION DE EMPRESAS,Presencial,...,"$700.000,00",Banco,,,,,,14141,27/05/2026,1033377195
58,07-30-2026 12:43:33,191.156.159.249,9870-88242954-260722093908,Vinculaciones David Alejandro Orjuela Chaparro,HECTOR ALEXANDER HIGUERA DE LOS REYES,Colombia - Boyaca - Tunja,88242954,Normal,INGENIERIA INDUSTRIAL,Virtual,...,"$83.674,00",Pse,,,,,,16391,30/07/2026,1025521082
65,07-30-2026 15:00:02,38.210.96.76,9889-16499517-260728010742,Vinculaciones Miryam Daniela Callejas Perez,Carlos Hernán Valencia Hurtado,Colombia - Valle Del Cauca - Buenaventura,16499517,Normal,ESPECIALIZACIÓN EN GERENCIA EDUCATIVA,Virtual,...,"$193.948,00",Banco,,,,,,16424,30/07/2026,1053616791
72,07-31-2026 10:45:51,38.210.96.72,9895-1122141658-260729021731,Vinculaciones Miryam Daniela Callejas Perez,Luis Alberto Cordoba Gonzalez,Colombia - Meta - Villavicencio,1122141658,Normal,ESPECIALIZACION EN CONTRATACION ESTATAL,Virtual,...,"$193.948,00",Pse,,,,,,16528,31/07/2026,1053616791
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56521,10-06-2026 22:05:36,191.156.18.44,10026-1044218402-261004060249,Vinculaciones Jesus David Jimenez Amado,RAFAEL EDUARDO ALVEAR MEDINA,Colombia - Atlantico - Barranquilla,1044218402,Normal,NEGOCIOS INTERNACIONALES,Virtual,...,"$42.400,00",Pse,,,,,,20870,6/10/2026,1081811110
56530,10-06-2026 22:23:19,191.156.18.44,9604-1066098610-260930061607,Vinculaciones Jesus David Jimenez Amado,Ingrid Gissela Carrero Carrascal,Colombia - Cesar - Curumani,1066098610,Normal,ESPECIALIZACION EN CONTRATACION ESTATAL,Virtual,...,"$154.360,00",Pse,,,,,,20871,6/10/2026,1081811110
56539,,,,,,,,,,,...,None,None,None,None,None,None,None,None,None,1019099501
56540,,,,,,,,,,,...,None,None,None,None,None,None,None,None,None,1019099501


# METAS POR PERIODO Y POR LIDER

In [326]:
df_metas_virtual = pd.read_excel(r'C:\Users\alejandro_arboledas\OneDrive - Corporación Unificada Nacional de Educación Superior - CUN\Documentos\PYTHON\VINCULACIONES\MODELO3\2026\METAS_202609.xlsx', sheet_name= periodo_virtual)
df_metas_virtual

,GESTOR DOCUMENTO,GESTOR CORREO,CARGO,Meta,LIDER,SEDE,LOCALIDAD
0,1143244777,katherine_bohorquez@cun.edu.co,Gestor de ventas Senior,16,SANDRA,BARRANQUILLA,BARRANQUILLA
1,1081796706,joel_bolano@cun.edu.co,Gestor de ventas Junior,17,SANDRA,BARRANQUILLA,BARRANQUILLA
2,1081811110,jesus_jimenez@cun.edu.co,Gestor de ventas Junior,17,SANDRA,BARRANQUILLA,BARRANQUILLA
3,44156610,claudia_guzmanbe@cun.edu.co,Gestor de ventas Junior,17,SANDRA,BARRANQUILLA,BARRANQUILLA
4,45550304,katherine_rosado@cun.edu.co,Gestor de ventas Senior,19,SANDRA,CARTAGENA,CARTAGENA
5,1235040095,rosa_ahumada@cun.edu.co,Gestor de ventas Nuevo,1,SANDRA,CARTAGENA,CARTAGENA
6,1143329915,ronald_villalbay@cun.edu.co,Gestor de ventas Junior,16,SANDRA,CARTAGENA,CARTAGENA
7,25800034,sandra_durango@cun.edu.co,Gestor de ventas Junior,16,SANDRA,CARTAGENA,CARTAGENA
8,1110479583,karen_rosell@cun.edu.co,Gestor de ventas Senior,16,SANDRA,IBAGUE,IBAGUE
9,1110587566,margarita_navarro@cun.edu.co,Gestor de ventas Junior,17,SANDRA,IBAGUE,IBAGUE


In [327]:
df_metas_virtual = df_metas_virtual[df_metas_virtual['LIDER'] == lider]
df_metas_virtual.shape

(48, 7)

In [328]:
df_metas_virtual['GESTOR CORREO'] = df_metas_virtual['GESTOR CORREO'].str.upper()
df_ventas_virtual = df_pre_ventas[df_pre_ventas['Correo Ejecutivo'].isin(df_metas_virtual['GESTOR CORREO'])].copy()
df_ventas_virtual = df_ventas_virtual[df_ventas_virtual['Periodo'].isin([periodo_virtual])]
df_ventas_virtual.shape

(521, 32)

# SE DEJAN SOLO LAS COLUMNAS NECESARIAS


In [329]:
columnas = ['Hora agregado', 'Numero de recibo','Dirección IP','Correo del estudiante','Teléfono del estudiante',
'Ciudad de residencia','Imagen del Pago Generado','% De Descuento',
'Medio de pago','Fuente de pionero','Campana','Fuente campana','Content','Sitio web', 'Tipo de inscripción',
'ID único','Fecha','Documento Ejecutivo_x','Número de aprobación de pago','Tipo de Financiación', 'Convenio', 'Valor pagado']

df_ventas_virtual.drop(columns=columnas, inplace=True)
df_ventas_virtual['Periodo'] = df_ventas_virtual['Periodo'].astype(str).str.strip()
df_ventas_virtual = df_ventas_virtual.rename(columns={'Documento Ejecutivo_y':'Documento Ejecutivo'})
df_ventas_virtual['Documento Ejecutivo'] = df_ventas_virtual['Documento Ejecutivo'].astype(str)
df_ventas_virtual['Número de Documento de estudiante'] = df_ventas_virtual['Número de Documento de estudiante'].astype(str)
df_ventas_virtual['Llave']=df_ventas_virtual['Número de Documento de estudiante']+df_ventas_virtual['Periodo']
df_ventas_virtual.shape


(521, 11)

# SE TRAEN LA MARCA DE PAGO Y LA FUERZA COMERCIAL

In [330]:
periodos_sql = ",".join(f"'{p}'" for p in periodos)

In [331]:
para_orden = df_pre_ventas['Número de Documento de estudiante'].unique()
valores_unicos_df1_orden = "', '".join(map(str, para_orden))
valores_unicos_df1_orden = f"'{valores_unicos_df1_orden}'"

In [332]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(f''' 
        SELECT distinct DOC_ALUM,PERIODO,ESTADO_PAGO,fuerzazoho as FuerzaComercialFinal, NUEVO, CLASE_ACTUAL, TIPO, PROGRAMA, EST_MATRICULADO
        FROM ZOHO.Base_Personas WITH(NOLOCK)
        WHERE PERIODO IN ({periodos_sql})
        and doc_alum in ({valores_unicos_df1_orden})
                        ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_recibos = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()
df_recibos.shape

(3975, 9)

In [333]:
df_recibos['LLAVE_RECIBOS'] = df_recibos['DOC_ALUM'].astype(str)+df_recibos['PERIODO']
df_recibos_virtual = df_recibos[df_recibos['PERIODO'] == periodo_virtual].copy()
df_recibos_virtual.shape

(1528, 10)

# SE CRUZA LOS RECIBOS CON EL REGISTRO DE VENTAS

In [334]:
df_recibos_virtual = df_recibos_virtual.merge(df_ventas_virtual[['Propietario del registro','Modalidad','Correo Ejecutivo', 'Regional',
       'Documento Ejecutivo', 'Llave']],how='inner',left_on='LLAVE_RECIBOS',right_on='Llave')
df_recibos_virtual.drop_duplicates(inplace=True)
df_recibos_virtual.shape

(519, 16)

In [335]:
#tratamiento de columnas
df_recibos_virtual['CLASE_ACTUAL'] = df_recibos_virtual['CLASE_ACTUAL'].astype(str).str.upper()
df_recibos_virtual['FuerzaComercialFinal'] = df_recibos_virtual['FuerzaComercialFinal'].str.upper()
df_recibos_virtual['NUEVO'] = df_recibos_virtual['NUEVO'].astype(str).str.upper()
df_recibos_virtual['ESTADO_PAGO'] = df_recibos_virtual['ESTADO_PAGO'].astype(str).str.upper()
df_recibos_virtual['FuerzaComercialFinal'] = df_recibos_virtual['FuerzaComercialFinal'].str.strip()
df_recibos_virtual['NUEVO'] = df_recibos_virtual['NUEVO'].str.strip()
df_recibos_virtual['CLASE_ACTUAL'] = df_recibos_virtual['CLASE_ACTUAL'].str.strip()
df_recibos_virtual['ESTADO_PAGO'] = df_recibos_virtual['ESTADO_PAGO'].str.strip()
df_recibos_virtual['EST_MATRICULADO'] = (
    df_recibos_virtual['EST_MATRICULADO']
    .astype(str)
    .str.strip()
    .str.upper()
)

In [336]:
# se divide entre validos
df_validos_virtual = df_recibos_virtual[
    ((df_recibos_virtual['FuerzaComercialFinal'] == 'BASICA Y MEDIA')  | (df_recibos_virtual['FuerzaComercialFinal'] == 'VINCULACIONES') | (df_recibos_virtual['FuerzaComercialFinal'] == 'INDETERMINADO')) &
    (df_recibos_virtual['NUEVO'] == 'NUEVO') &
    (df_recibos_virtual['EST_MATRICULADO'] == '1-ACTIVO') & 
    (df_recibos_virtual['ESTADO_PAGO'] == 'PAGO')
].copy()
df_validos_virtual.drop_duplicates(inplace=True)
df_validos_virtual.shape

(494, 16)

In [337]:
# se divide entre no validos
mask = (
    df_recibos_virtual['FuerzaComercialFinal'].isin(
        ['BASICA Y MEDIA', 'VINCULACIONES', 'INDETERMINADO']
    ) &
    (df_recibos_virtual['NUEVO'] == 'NUEVO') &
    (df_recibos_virtual['EST_MATRICULADO'] == '1-ACTIVO') &
    (df_recibos_virtual['ESTADO_PAGO'] == 'PAGO')
)
df_no_validos_virtual = df_recibos_virtual[~mask].copy()
df_no_validos_virtual.drop_duplicates(inplace=True)
df_no_validos_virtual.shape

(25, 16)

In [338]:
# Inicializar MOTIVO vacío
df_no_validos_virtual['MOTIVO'] = ''

# Condición 1
condicion = df_no_validos_virtual['ESTADO_PAGO'] == 'NO PAGO'
df_no_validos_virtual.loc[condicion, 'MOTIVO'] += 'NO REGISTRA PAGO | '

# Condición 2
condicion = (
    (df_no_validos_virtual['NUEVO'] != 'NUEVO')
)
df_no_validos_virtual.loc[condicion, 'MOTIVO'] += 'NO ES UN ESTUDIANTE NUEVO | '

# Condición 3
condicion = df_no_validos_virtual['FuerzaComercialFinal'] != 'VINCULACIONES'
df_no_validos_virtual.loc[condicion, 'MOTIVO'] += 'PERTENECE A OTRA FUERZA | '

# Condición 4
condicion = df_no_validos_virtual['EST_MATRICULADO'] != '1-ACTIVO'
df_no_validos_virtual.loc[condicion, 'MOTIVO'] += 'MATRICULA INACTIVA | '

# Quitar el último " | "
df_no_validos_virtual['MOTIVO'] = (
    df_no_validos_virtual['MOTIVO']
    .str.rstrip(' |')
)

# Los que no tengan ningún motivo quedan como N/A
df_no_validos_virtual.loc[
    df_no_validos_virtual['MOTIVO'] == '',
    'MOTIVO'
] = 'N/A'

df_no_validos_virtual

,DOC_ALUM,PERIODO,ESTADO_PAGO,FuerzaComercialFinal,NUEVO,CLASE_ACTUAL,TIPO,PROGRAMA,EST_MATRICULADO,LLAVE_RECIBOS,Propietario del registro,Modalidad,Correo Ejecutivo,Regional,Documento Ejecutivo,Llave,MOTIVO
15,1002600121,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,INGENIERÍA DE SISTEMAS,1-ACTIVO,100260012126V05,Vinculaciones Miryam Daniela Callejas Perez,Virtual,MIRYAM_CALLEJASP@CUN.EDU.CO,Tunja,1053616791,100260012126V05,NO REGISTRA PAGO
46,1004612618,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS DE PROGRAMACIÓ...,1-ACTIVO,100461261826V05,Vinculaciones Edwin Alejandro Gomez Montenegro,Virtual,EDWIN_GOMEZM@CUN.EDU.CO,Cali,1143966728,100461261826V05,NO REGISTRA PAGO
62,1005912112,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS DE PROGRAMACIÓ...,1-ACTIVO,100591211226V05,Vinculaciones Nicolas Viafara Ospina,Virtual,NICOLAS_VIAFARAOS@CUN.EDU.CO,Ibague,1112496766,100591211226V05,NO REGISTRA PAGO
73,1006510343,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS OPERATIVOS DE ...,1-ACTIVO,100651034326V05,Vinculaciones Luis Alejandro Muñoz,Virtual,LUIS_MUNOZMU@CUN.EDU.CO,Pereira,1023938622,100651034326V05,NO REGISTRA PAGO
122,1023938622,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN OPERACIÓN DE PROCESOS D...,1-ACTIVO,102393862226V05,Vinculaciones Luis Alejandro Muñoz,Virtual,LUIS_MUNOZMU@CUN.EDU.CO,Pereira,1023938622,102393862226V05,NO REGISTRA PAGO
131,10306598,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN SOPORTE DE SISTEMAS E I...,1-ACTIVO,1030659826V05,Vinculaciones Fabio Andres Miranda Sequea,Virtual,FABIO_MIRANDA@CUN.EDU.CO,Sincelejo,1102877864,1030659826V05,NO REGISTRA PAGO
134,1025530850,26V05,PAGO,CONTACT,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS DE PROGRAMACIÓ...,1-ACTIVO,102553085026V05,Vinculaciones Deybhor Sveyder Peña Patiño,Virtual,DEYBHOR_PENAP@CUN.EDU.CO,Bucaramanga,1098773128,102553085026V05,PERTENECE A OTRA FUERZA
176,1043588897,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN SOPORTE DE SISTEMAS E I...,1-ACTIVO,104358889726V05,Vinculaciones Katherine Hellen Bohorquez Palencia,Virtual,KATHERINE_BOHORQUEZ@CUN.EDU.CO,Barranquilla,1143244777,104358889726V05,NO REGISTRA PAGO
191,1057411128,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN SOPORTE DE SISTEMAS E I...,1-ACTIVO,105741112826V05,Vinculaciones David Alejandro Orjuela Chaparro,Virtual,DAVID_ORJUELAC@CUN.EDU.CO,Tunja,1025521082,105741112826V05,NO REGISTRA PAGO
201,1063285933,26V05,NO PAGO,VINCULACIONES,NUEVO,REINTEGRO,Pregrado,TÉCNICA PROFESIONAL EN SOPORTE DE SISTEMAS E I...,1-ACTIVO,106328593326V05,Vinculaciones Miguel Angel Palomino Peralta,Virtual,MIGUEL_PALOMINO@CUN.EDU.CO,Montería,1128053909,106328593326V05,NO REGISTRA PAGO


In [339]:
# se traen las ventas que no tienen marca de pago
df_no_recibos_virtual = df_ventas_virtual[~df_ventas_virtual['Llave'].isin(df_recibos_virtual['LLAVE_RECIBOS'])].copy()
df_no_recibos_virtual['DOC_ALUM'] = df_no_recibos_virtual['Número de Documento de estudiante'].astype(str)
df_no_recibos_virtual['PERIODO'] = df_no_recibos_virtual['Periodo']
df_no_recibos_virtual['ESTADO_PAGO'] = ' '
df_no_recibos_virtual['EST_MATRICULADO'] = ' '
df_no_recibos_virtual['FuerzaComercialFinal'] = ' '
df_no_recibos_virtual['NUEVO'] = ' '
df_no_recibos_virtual['CLASE_ACTUAL'] = ' '
df_no_recibos_virtual['TIPO'] = ' '
df_no_recibos_virtual['PROGRAMA'] = df_no_recibos_virtual['Programa vendido'] 
df_no_recibos_virtual['LLAVE_RECIBOS'] = ' '
df_no_recibos_virtual['MOTIVO'] = 'NO CUENTA CON MARCA DE PAGO'
df_no_recibos_virtual = df_no_recibos_virtual[['DOC_ALUM', 'PERIODO', 'ESTADO_PAGO', 'FuerzaComercialFinal', 'NUEVO',
       'CLASE_ACTUAL', 'TIPO', 'PROGRAMA', 'EST_MATRICULADO', 'LLAVE_RECIBOS',
       'Propietario del registro', 'Modalidad', 'Correo Ejecutivo', 'Regional',
       'Documento Ejecutivo', 'Llave','MOTIVO']]
df_no_recibos_virtual.shape

(5, 17)

In [340]:
# SE UNIFICAN LOS NO VALIDOS Y LOS QUE NO TIENEN MARCA DE PAGO
df_no_validos_2 = pd.concat([df_no_validos_virtual,df_no_recibos_virtual])
df_no_validos_2

,DOC_ALUM,PERIODO,ESTADO_PAGO,FuerzaComercialFinal,NUEVO,CLASE_ACTUAL,TIPO,PROGRAMA,EST_MATRICULADO,LLAVE_RECIBOS,Propietario del registro,Modalidad,Correo Ejecutivo,Regional,Documento Ejecutivo,Llave,MOTIVO
15,1002600121,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,INGENIERÍA DE SISTEMAS,1-ACTIVO,100260012126V05,Vinculaciones Miryam Daniela Callejas Perez,Virtual,MIRYAM_CALLEJASP@CUN.EDU.CO,Tunja,1053616791,100260012126V05,NO REGISTRA PAGO
46,1004612618,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS DE PROGRAMACIÓ...,1-ACTIVO,100461261826V05,Vinculaciones Edwin Alejandro Gomez Montenegro,Virtual,EDWIN_GOMEZM@CUN.EDU.CO,Cali,1143966728,100461261826V05,NO REGISTRA PAGO
62,1005912112,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS DE PROGRAMACIÓ...,1-ACTIVO,100591211226V05,Vinculaciones Nicolas Viafara Ospina,Virtual,NICOLAS_VIAFARAOS@CUN.EDU.CO,Ibague,1112496766,100591211226V05,NO REGISTRA PAGO
73,1006510343,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS OPERATIVOS DE ...,1-ACTIVO,100651034326V05,Vinculaciones Luis Alejandro Muñoz,Virtual,LUIS_MUNOZMU@CUN.EDU.CO,Pereira,1023938622,100651034326V05,NO REGISTRA PAGO
122,1023938622,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN OPERACIÓN DE PROCESOS D...,1-ACTIVO,102393862226V05,Vinculaciones Luis Alejandro Muñoz,Virtual,LUIS_MUNOZMU@CUN.EDU.CO,Pereira,1023938622,102393862226V05,NO REGISTRA PAGO
131,10306598,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN SOPORTE DE SISTEMAS E I...,1-ACTIVO,1030659826V05,Vinculaciones Fabio Andres Miranda Sequea,Virtual,FABIO_MIRANDA@CUN.EDU.CO,Sincelejo,1102877864,1030659826V05,NO REGISTRA PAGO
134,1025530850,26V05,PAGO,CONTACT,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS DE PROGRAMACIÓ...,1-ACTIVO,102553085026V05,Vinculaciones Deybhor Sveyder Peña Patiño,Virtual,DEYBHOR_PENAP@CUN.EDU.CO,Bucaramanga,1098773128,102553085026V05,PERTENECE A OTRA FUERZA
176,1043588897,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN SOPORTE DE SISTEMAS E I...,1-ACTIVO,104358889726V05,Vinculaciones Katherine Hellen Bohorquez Palencia,Virtual,KATHERINE_BOHORQUEZ@CUN.EDU.CO,Barranquilla,1143244777,104358889726V05,NO REGISTRA PAGO
191,1057411128,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN SOPORTE DE SISTEMAS E I...,1-ACTIVO,105741112826V05,Vinculaciones David Alejandro Orjuela Chaparro,Virtual,DAVID_ORJUELAC@CUN.EDU.CO,Tunja,1025521082,105741112826V05,NO REGISTRA PAGO
201,1063285933,26V05,NO PAGO,VINCULACIONES,NUEVO,REINTEGRO,Pregrado,TÉCNICA PROFESIONAL EN SOPORTE DE SISTEMAS E I...,1-ACTIVO,106328593326V05,Vinculaciones Miguel Angel Palomino Peralta,Virtual,MIGUEL_PALOMINO@CUN.EDU.CO,Montería,1128053909,106328593326V05,NO REGISTRA PAGO


# SE TRAEN LAS OPORTUNIDADES DE ZOHO CRM

In [341]:
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'

try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    df_zoho = pd.read_sql(f'''
Select distinct 
o.Propietario_de_Oportunidad_Name,
o.Hora_de_creación,
o.Fecha_de_cierre,
o.Fase,
o.Hora_de_modificación,
o.Número_de_Documento,
o.Programa_de_interés,
o.PERIODO,
o.Hora_de_la_última_actividad,
u.Correo_electrónico
from zoho.CRM.Oportunidades o
left join zoho.CRM.Usuarios U
ON o.Propietario_de_Oportunidad = u.Id
    where Periodo in ({periodos_sql})
and Fase LIKE '%Matriculado%'
                               ''', engine2_1)
    Consulta_z2 = df_zoho.copy()
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")
df_zoho.shape

Conexion a la base de datos realizada
Conexión cerrada


(14808, 10)

In [342]:
df_zoho['Correo_electrónico'] = df_zoho['Correo_electrónico'].str.upper()
df_zoho['LLAVE_ZOHO']=df_zoho['Número_de_Documento'].astype(str)+df_zoho['PERIODO'].astype(str).str.strip()+df_zoho['Correo_electrónico']
df_zoho_vinculaciones = df_zoho[df_zoho['Correo_electrónico'].isin(df_pre_ventas['Correo Ejecutivo'])].copy()
df_zoho_vinculaciones = df_zoho_vinculaciones.drop_duplicates()
df_zoho_vinculaciones.shape

(3712, 11)

In [343]:
# se crea la llave para cruzar por doc alum+periodo+correo ejecutivo
df_validos_virtual['LLAVE_ZOHO'] = df_validos_virtual['DOC_ALUM']+df_validos_virtual['PERIODO']+df_validos_virtual['Correo Ejecutivo']

In [344]:
df_zoho_recibos_virtual = df_validos_virtual.merge(df_zoho_vinculaciones[['Propietario_de_Oportunidad_Name','Fase','Hora_de_modificación','LLAVE_ZOHO']],left_on='LLAVE_ZOHO',right_on='LLAVE_ZOHO',how='inner').copy()
df_zoho_recibos_virtual.drop_duplicates(inplace=True)
df_zoho_recibos_virtual.shape

(467, 20)

In [345]:
df_no_validos_zoho = df_validos_virtual[~df_validos_virtual['LLAVE_ZOHO'].isin(df_zoho_vinculaciones['LLAVE_ZOHO'])].copy()
df_no_validos_zoho['MOTIVO'] = 'NO ESTA OK EN ZOHO'
df_no_validos_zoho = df_no_validos_zoho.drop(columns=['LLAVE_ZOHO'])
df_no_validos_zoho.shape

(28, 17)

In [346]:
# SE UNIFICAN LOS NO VALIDOS TOTALES DE VIRTUALES
df_no_validos_3 = pd.concat([df_no_validos_2,df_no_validos_zoho])
df_no_validos_3

,DOC_ALUM,PERIODO,ESTADO_PAGO,FuerzaComercialFinal,NUEVO,CLASE_ACTUAL,TIPO,PROGRAMA,EST_MATRICULADO,LLAVE_RECIBOS,Propietario del registro,Modalidad,Correo Ejecutivo,Regional,Documento Ejecutivo,Llave,MOTIVO
15,1002600121,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,INGENIERÍA DE SISTEMAS,1-ACTIVO,100260012126V05,Vinculaciones Miryam Daniela Callejas Perez,Virtual,MIRYAM_CALLEJASP@CUN.EDU.CO,Tunja,1053616791,100260012126V05,NO REGISTRA PAGO
46,1004612618,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS DE PROGRAMACIÓ...,1-ACTIVO,100461261826V05,Vinculaciones Edwin Alejandro Gomez Montenegro,Virtual,EDWIN_GOMEZM@CUN.EDU.CO,Cali,1143966728,100461261826V05,NO REGISTRA PAGO
62,1005912112,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS DE PROGRAMACIÓ...,1-ACTIVO,100591211226V05,Vinculaciones Nicolas Viafara Ospina,Virtual,NICOLAS_VIAFARAOS@CUN.EDU.CO,Ibague,1112496766,100591211226V05,NO REGISTRA PAGO
73,1006510343,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS OPERATIVOS DE ...,1-ACTIVO,100651034326V05,Vinculaciones Luis Alejandro Muñoz,Virtual,LUIS_MUNOZMU@CUN.EDU.CO,Pereira,1023938622,100651034326V05,NO REGISTRA PAGO
122,1023938622,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN OPERACIÓN DE PROCESOS D...,1-ACTIVO,102393862226V05,Vinculaciones Luis Alejandro Muñoz,Virtual,LUIS_MUNOZMU@CUN.EDU.CO,Pereira,1023938622,102393862226V05,NO REGISTRA PAGO
131,10306598,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN SOPORTE DE SISTEMAS E I...,1-ACTIVO,1030659826V05,Vinculaciones Fabio Andres Miranda Sequea,Virtual,FABIO_MIRANDA@CUN.EDU.CO,Sincelejo,1102877864,1030659826V05,NO REGISTRA PAGO
134,1025530850,26V05,PAGO,CONTACT,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS DE PROGRAMACIÓ...,1-ACTIVO,102553085026V05,Vinculaciones Deybhor Sveyder Peña Patiño,Virtual,DEYBHOR_PENAP@CUN.EDU.CO,Bucaramanga,1098773128,102553085026V05,PERTENECE A OTRA FUERZA
176,1043588897,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN SOPORTE DE SISTEMAS E I...,1-ACTIVO,104358889726V05,Vinculaciones Katherine Hellen Bohorquez Palencia,Virtual,KATHERINE_BOHORQUEZ@CUN.EDU.CO,Barranquilla,1143244777,104358889726V05,NO REGISTRA PAGO
191,1057411128,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN SOPORTE DE SISTEMAS E I...,1-ACTIVO,105741112826V05,Vinculaciones David Alejandro Orjuela Chaparro,Virtual,DAVID_ORJUELAC@CUN.EDU.CO,Tunja,1025521082,105741112826V05,NO REGISTRA PAGO
201,1063285933,26V05,NO PAGO,VINCULACIONES,NUEVO,REINTEGRO,Pregrado,TÉCNICA PROFESIONAL EN SOPORTE DE SISTEMAS E I...,1-ACTIVO,106328593326V05,Vinculaciones Miguel Angel Palomino Peralta,Virtual,MIGUEL_PALOMINO@CUN.EDU.CO,Montería,1128053909,106328593326V05,NO REGISTRA PAGO


# SE CRUZA CON EL HISTORICO 

In [347]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 
            select * from CUN_REPOSITORIO.comisiones.COMISIONES_VINCULACIONES_ACTUALES
            WHERE [COMISION TOTAL] != 0
 ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_historico = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()
df_historico.shape

(3407, 33)

In [348]:
df_historico['LLAVE_HISTORICO'] = df_historico['Documento_Estudiante'].astype(str) + df_historico['PERIODO'].astype(str)

# SE DEJA TODOS LOS VALIDOS EN UN SOLO DATAFRAME

In [349]:
df_validos_virtuales_final = df_zoho_recibos_virtual[~df_zoho_recibos_virtual['Llave'].isin(df_historico['LLAVE_HISTORICO'])].copy()
df_validos_virtuales_final.shape

(467, 20)

# SE DEJA TODOS LOS NO VALIDOS VIRTUALES EN UN SOLO DATAFRAME

In [350]:
df_no_validos_historico = df_zoho_recibos_virtual[df_zoho_recibos_virtual['Llave'].isin(df_historico['LLAVE_HISTORICO'])].copy()
df_no_validos_historico['MOTIVO'] = 'YA SE PAGO EN LA LIQUIDACION PASADA' 
df_no_validos_historico = df_no_validos_historico.drop(columns=['LLAVE_ZOHO','Propietario_de_Oportunidad_Name','Fase','Hora_de_modificación'])
df_no_validos_historico.shape

(0, 17)

In [351]:
df_novalidos_virtual_final = pd.concat([df_no_validos_3,df_no_validos_historico])
df_novalidos_virtual_final.shape

(58, 17)

# CALCULOS

In [352]:
# Filtrar y contar las ventas por estado de pago (PAGO y NO PAGO)
ventas_asesor_virtuales = (
    df_validos_virtuales_final
    .groupby(["Documento Ejecutivo"])
    .size()
    .reset_index(name="VENTAS_VIRTUALES")
).copy()


In [353]:
df_metas_virtual['GESTOR DOCUMENTO'] = df_metas_virtual['GESTOR DOCUMENTO'].astype(str)
ventas_asesor_virtuales['Documento Ejecutivo'] = ventas_asesor_virtuales['Documento Ejecutivo'].astype(str)
ventas_asesor_virtuales = ventas_asesor_virtuales.merge(df_metas_virtual,how='left',left_on='Documento Ejecutivo',right_on='GESTOR DOCUMENTO').copy()
ventas_asesor_virtuales = ventas_asesor_virtuales.drop(columns=['GESTOR DOCUMENTO'])
ventas_asesor_virtuales

,Documento Ejecutivo,VENTAS_VIRTUALES,GESTOR CORREO,CARGO,Meta,LIDER,SEDE,LOCALIDAD
0,1000224303,2,GUISELLE_VALENZUELA@CUN.EDU.CO,Gestor de ventas Nuevo,1.0,SANDRA,IBAGUE,IBAGUE
1,1001344725,4,JULIANA_AUDOR@CUN.EDU.CO,Gestor de ventas Nuevo,1.0,SANDRA,NEIVA,NEIVA
2,1004734611,3,NaN,NaN,NaN,NaN,NaN,NaN
3,1005337691,7,KAREN_AGUILAR@CUN.EDU.CO,Gestor de ventas Junior,21.0,SANDRA,BUCARAMANGA,BUCARAMANGA
4,1006053851,1,VALENTINA_MONTOYAC@CUN.EDU.CO,Gestor de ventas Junior,21.0,SANDRA,CALI,CALI
5,1023938622,6,LUIS_MUNOZMU@CUN.EDU.CO,Gestor de ventas Junior,21.0,SANDRA,PEREIRA,PEREIRA
6,1025521082,4,DAVID_ORJUELAC@CUN.EDU.CO,Gestor de ventas Junior,21.0,SANDRA,TUNJA,TUNJA
7,1033377195,1,LAURA_BEDOYAC@CUN.EDU.CO,Gestor de ventas Junior,17.0,SANDRA,MEDELLIN,MEDELLIN
8,1036672351,17,JUAN_BRANPE@CUN.EDU.CO,Gestor de ventas Junior,17.0,SANDRA,MEDELLIN,MEDELLIN
9,1053616791,3,MIRYAM_CALLEJASP@CUN.EDU.CO,Gestor de ventas Junior,21.0,SANDRA,TUNJA,TUNJA


# SE DEFINE EL ESQUEMA COMISIONAL

In [354]:
esquema_comisiones_virtuales = {

    "SANDRA": {
        "BARRANQUILLA": {
            "Gestor de ventas Senior":{ 
                        16: [  
                                (16, 30, 90000),
                                (31, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        17: [  
                                (17, 30, 90000),
                                (31, float('inf'), 120000)
                            
                            ]       
            }},

        "CARTAGENA": {
            "Gestor de ventas Senior":{ 
                        19: [  
                                (19, 30, 100000),
                                (31, float('inf'), 130000)
                            ]},
            "Gestor de ventas Junior":{ 
                        16: [  
                                (16, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ]       
            },
            "Gestor de ventas Nuevo":{ 
                        1: [  
                                (1, 5, 23000),
                                (5, 7, 100000),
                                (8, float('inf'), 130000)
                            
                            ]       
            }},
        "IBAGUE": {
            "Gestor de ventas Senior":{ 
                        16: [  
                                (16, 30, 90000),
                                (31, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        17: [  
                                (17, 30, 90000),
                                (31, float('inf'), 120000)
                            
                            ]       
            },
            "Gestor de ventas Nuevo":{ 
                        1: [  
                                (1, 30, 23000),
                                (30, 37, 90000),
                                (38, float('inf'), 120000)
                            
                            ]       
            }},
        "MEDELLIN": {
            "Gestor de ventas Senior":{ 
                        16: [  
                                (16, 30, 90000),
                                (31, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        17: [  
                                (17, 30, 90000),
                                (31, float('inf'), 120000)
                            
                            ]       
            }},
        "MONTERIA": {
            "Gestor de ventas Senior":{ 
                        16: [  
                                (16, 30, 90000),
                                (31, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        17: [  
                                (17, 30, 90000),
                                (31, float('inf'), 120000)
                            
                            ]       
            }},
        "NEIVA": {
            "Gestor de ventas Senior":{ 
                        16: [  
                                (16, 30, 90000),
                                (31, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        17: [  
                                (17, 30, 90000),
                                (31, float('inf'), 120000)
                            
                            ]       
            },
            "Gestor de ventas Nuevo":{ 
                        1: [  
                                (1, 5, 23000),
                                (5, 7, 90000),
                                (8, float('inf'), 120000)
                            
                            ]       
            }},
        "SANTA MARTA": {
            "Gestor de ventas Senior":{ 
                        16: [  
                                (16, 30, 90000),
                                (31, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        17: [  
                                (17, 30, 90000),
                                (31, float('inf'), 120000)
                            
                            ]       
            }},
        "SINCELEJO": {
            "Gestor de ventas Senior":{ 
                        16: [  
                                (16, 30, 90000),
                                (31, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        17: [  
                                (17, 30, 90000),
                                (31, float('inf'), 120000)
                            
                            ]       
            }},               
        "CALI": {
            "Gestor de ventas Senior":{ 
                        19: [  
                                (19, 30, 100000),
                                (31, float('inf'), 130000)
                            ]},
            "Gestor de ventas Junior":{ 
                        21: [  
                                (21, 30, 100000),
                                (31, float('inf'), 130000)],
                        16: [  
                                (16, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ]            
            }}, 
        "PEREIRA": {
            "Gestor de ventas Senior":{ 
                        19: [  
                                (19, 30, 100000),
                                (31, float('inf'), 130000)
                            ]},
            "Gestor de ventas Junior":{ 
                        21: [  
                                (21, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ],
                        16: [  
                                (16, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ]     
                            }},
        "BUCARAMANGA": {
            "Gestor de ventas Senior":{ 
                        19: [  
                                (19, 30, 100000),
                                (31, float('inf'), 130000)
                            ]},
            "Gestor de ventas Junior":{ 
                        21: [  
                                (21, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ],
                        16: [  
                                (16, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ]     }},
         "TUNJA": {
            "Gestor de ventas Senior":{ 
                        17: [  
                                (17, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ]  },
            "Gestor de ventas Junior":{ 
                        21: [  
                                (21, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ]       
            }}            
    }}

In [355]:
def asignar_comision_virtuales(lider, sede, cargo, meta, ventas):

    # 1️⃣ Validar líder
    if lider not in esquema_comisiones_virtuales:
        return 0

    # 2️⃣ Validar sede
    if sede not in esquema_comisiones_virtuales[lider]:
        return 0

    # 3️⃣ Validar cargo
    if cargo not in esquema_comisiones_virtuales[lider][sede]:
        return 0

    # 4️⃣ Validar meta
    if meta not in esquema_comisiones_virtuales[lider][sede][cargo]:
        return 0

    # 5️⃣ Validar cumplimiento de meta
    if ventas < meta:
        return 0

    # 6️⃣ Obtener los pisos correspondientes a la meta
    rangos = esquema_comisiones_virtuales[lider][sede][cargo][meta]

    # 7️⃣ Buscar el piso correspondiente según las ventas
    for min_rango, max_rango, comision in rangos:

        if min_rango <= ventas <= max_rango:

            # Comisión por venta × número de ventas
            return ventas * comision

    return 0

In [356]:
ventas_asesor_virtuales["COMISION_TOTAL_VIRTUAL"] = ventas_asesor_virtuales.apply(
    lambda row: asignar_comision_virtuales(
        row["LIDER"],
        row["SEDE"],
        row["CARGO"],
        row["Meta"],
        row["VENTAS_VIRTUALES"]
    ),
    axis=1
)
ventas_asesor_virtuales['COMISION_UNITARIA_VIRTUAL'] = ventas_asesor_virtuales['COMISION_TOTAL_VIRTUAL'] / ventas_asesor_virtuales['VENTAS_VIRTUALES']
ventas_asesor_virtuales['%CUMPLIMIENTO_VIRTUAL'] = (
    (ventas_asesor_virtuales['VENTAS_VIRTUALES'] / ventas_asesor_virtuales['Meta']) * 100).round(2).astype(str) + '%'
ventas_asesor_virtuales

,Documento Ejecutivo,VENTAS_VIRTUALES,GESTOR CORREO,CARGO,Meta,LIDER,SEDE,LOCALIDAD,COMISION_TOTAL_VIRTUAL,COMISION_UNITARIA_VIRTUAL,%CUMPLIMIENTO_VIRTUAL
0,1000224303,2,GUISELLE_VALENZUELA@CUN.EDU.CO,Gestor de ventas Nuevo,1.0,SANDRA,IBAGUE,IBAGUE,46000,23000.0,200.0%
1,1001344725,4,JULIANA_AUDOR@CUN.EDU.CO,Gestor de ventas Nuevo,1.0,SANDRA,NEIVA,NEIVA,92000,23000.0,400.0%
2,1004734611,3,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,nan%
3,1005337691,7,KAREN_AGUILAR@CUN.EDU.CO,Gestor de ventas Junior,21.0,SANDRA,BUCARAMANGA,BUCARAMANGA,0,0.0,33.33%
4,1006053851,1,VALENTINA_MONTOYAC@CUN.EDU.CO,Gestor de ventas Junior,21.0,SANDRA,CALI,CALI,0,0.0,4.76%
5,1023938622,6,LUIS_MUNOZMU@CUN.EDU.CO,Gestor de ventas Junior,21.0,SANDRA,PEREIRA,PEREIRA,0,0.0,28.57%
6,1025521082,4,DAVID_ORJUELAC@CUN.EDU.CO,Gestor de ventas Junior,21.0,SANDRA,TUNJA,TUNJA,0,0.0,19.05%
7,1033377195,1,LAURA_BEDOYAC@CUN.EDU.CO,Gestor de ventas Junior,17.0,SANDRA,MEDELLIN,MEDELLIN,0,0.0,5.88%
8,1036672351,17,JUAN_BRANPE@CUN.EDU.CO,Gestor de ventas Junior,17.0,SANDRA,MEDELLIN,MEDELLIN,1530000,90000.0,100.0%
9,1053616791,3,MIRYAM_CALLEJASP@CUN.EDU.CO,Gestor de ventas Junior,21.0,SANDRA,TUNJA,TUNJA,0,0.0,14.29%


## PROCESO DE COMISION PARA PRESENCIALES
-- SE DEFINEN LAS METAS

In [357]:
df_metas_presencial = pd.read_excel(r'C:\Users\alejandro_arboledas\OneDrive - Corporación Unificada Nacional de Educación Superior - CUN\Documentos\PYTHON\VINCULACIONES\MODELO3\2026\METAS_202609.xlsx', sheet_name= periodo_presencial)
df_metas_presencial

,GESTOR DOCUMENTO,GESTOR CORREO,CARGO,Meta,LIDER,SEDE,LOCALIDAD
0,1143244777,katherine_bohorquez@cun.edu.co,Gestor de ventas Senior,16,SANDRA,BARRANQUILLA,BARRANQUILLA
1,1081796706,joel_bolano@cun.edu.co,Gestor de ventas Junior,17,SANDRA,BARRANQUILLA,BARRANQUILLA
2,1081811110,jesus_jimenez@cun.edu.co,Gestor de ventas Junior,17,SANDRA,BARRANQUILLA,BARRANQUILLA
3,44156610,claudia_guzmanbe@cun.edu.co,Gestor de ventas Junior,17,SANDRA,BARRANQUILLA,BARRANQUILLA
4,45550304,katherine_rosado@cun.edu.co,Gestor de ventas Senior,0,SANDRA,CARTAGENA,CARTAGENA
5,1235040095,rosa_ahumada@cun.edu.co,Gestor de ventas Nuevo,0,SANDRA,CARTAGENA,CARTAGENA
6,1143329915,ronald_villalbay@cun.edu.co,Gestor de ventas Junior,0,SANDRA,CARTAGENA,CARTAGENA
7,25800034,sandra_durango@cun.edu.co,Gestor de ventas Junior,0,SANDRA,CARTAGENA,CARTAGENA
8,1110479583,karen_rosell@cun.edu.co,Gestor de ventas Senior,16,SANDRA,IBAGUE,IBAGUE
9,1110587566,margarita_navarro@cun.edu.co,Gestor de ventas Junior,18,SANDRA,IBAGUE,IBAGUE


In [358]:
df_metas_presencial = df_metas_presencial[df_metas_presencial['LIDER'] == lider]
df_metas_presencial.shape

(48, 7)

VENTAS PRESENCIALES

In [359]:
df_metas_presencial['GESTOR CORREO'] = df_metas_presencial['GESTOR CORREO'].str.upper()
df_ventas_presencial = df_pre_ventas[df_pre_ventas['Correo Ejecutivo'].isin(df_metas_presencial['GESTOR CORREO'])].copy()
df_ventas_presencial = df_ventas_presencial[df_ventas_presencial['Periodo'].isin([periodo_presencial])]
df_ventas_presencial.shape

(285, 32)

In [360]:
columnas = ['Hora agregado', 'Numero de recibo','Dirección IP','Correo del estudiante','Teléfono del estudiante',
'Ciudad de residencia','Imagen del Pago Generado','% De Descuento',
'Medio de pago','Fuente de pionero','Campana','Fuente campana','Content','Sitio web', 'Tipo de inscripción',
'ID único','Fecha','Documento Ejecutivo_x','Número de aprobación de pago','Tipo de Financiación', 'Convenio', 'Valor pagado']

df_ventas_presencial.drop(columns=columnas, inplace=True)
df_ventas_presencial['Periodo'] = df_ventas_presencial['Periodo'].astype(str).str.strip()
df_ventas_presencial = df_ventas_presencial.rename(columns={'Documento Ejecutivo_y':'Documento Ejecutivo'})
df_ventas_presencial['Documento Ejecutivo'] = df_ventas_presencial['Documento Ejecutivo'].astype(str)
df_ventas_presencial['Número de Documento de estudiante'] = df_ventas_presencial['Número de Documento de estudiante'].astype(str)
df_ventas_presencial['Llave']=df_ventas_presencial['Número de Documento de estudiante']+df_ventas_presencial['Periodo']
df_ventas_presencial.shape

(285, 11)

SE CRUZA CON RECIBOS

In [361]:
df_recibos_presencial = df_recibos[df_recibos['PERIODO'] == periodo_presencial].copy()
df_recibos_presencial.shape

(344, 10)

In [362]:
df_recibos_presencial = df_recibos_presencial.merge(df_ventas_presencial[['Propietario del registro','Modalidad','Correo Ejecutivo', 'Regional',
       'Documento Ejecutivo', 'Llave']],how='inner',left_on='LLAVE_RECIBOS',right_on='Llave')
df_recibos_presencial.drop_duplicates(inplace=True)
df_recibos_presencial.shape

(231, 16)

In [363]:
#tratamiento de columnas
df_recibos_presencial['CLASE_ACTUAL'] = df_recibos_presencial['CLASE_ACTUAL'].astype(str).str.upper()
df_recibos_presencial['FuerzaComercialFinal'] = df_recibos_presencial['FuerzaComercialFinal'].str.upper()
df_recibos_presencial['NUEVO'] = df_recibos_presencial['NUEVO'].astype(str).str.upper()
df_recibos_presencial['ESTADO_PAGO'] = df_recibos_presencial['ESTADO_PAGO'].astype(str).str.upper()
df_recibos_presencial['FuerzaComercialFinal'] = df_recibos_presencial['FuerzaComercialFinal'].str.strip()
df_recibos_presencial['NUEVO'] = df_recibos_presencial['NUEVO'].str.strip()
df_recibos_presencial['CLASE_ACTUAL'] = df_recibos_presencial['CLASE_ACTUAL'].str.strip()
df_recibos_presencial['ESTADO_PAGO'] = df_recibos_presencial['ESTADO_PAGO'].str.strip()
df_recibos_presencial['EST_MATRICULADO'] = (
    df_recibos_presencial['EST_MATRICULADO']
    .astype(str)
    .str.strip()
    .str.upper()
)

In [364]:
# se divide entre validos
df_validos_presencial = df_recibos_presencial[
    ((df_recibos_presencial['FuerzaComercialFinal'] == 'BASICA Y MEDIA') | (df_recibos_presencial['FuerzaComercialFinal'] == 'VINCULACIONES') | (df_recibos_presencial['FuerzaComercialFinal'] == 'INDETERMINADO')) &
    (df_recibos_presencial['NUEVO'] == 'NUEVO') &
    (df_recibos_presencial['EST_MATRICULADO'] == '1-ACTIVO') & 
    (df_recibos_presencial['ESTADO_PAGO'] == 'PAGO')
].copy()
df_validos_presencial.drop_duplicates(inplace=True)
df_validos_presencial.shape 

(228, 16)

In [365]:
# se divide entre no validos
mask = (
    df_recibos_presencial['FuerzaComercialFinal'].isin(
        ['BASICA Y MEDIA', 'VINCULACIONES', 'INDETERMINADO']
    ) &
    (df_recibos_presencial['NUEVO'] == 'NUEVO') &
    (df_recibos_presencial['EST_MATRICULADO'] == '1-ACTIVO') &
    (df_recibos_presencial['ESTADO_PAGO'] == 'PAGO')
)
df_no_validos_presencial = df_recibos_presencial[~mask].copy()
df_no_validos_presencial.drop_duplicates(inplace=True)
df_no_validos_presencial.shape

(3, 16)

In [366]:
# Inicializar MOTIVO vacío
df_no_validos_presencial['MOTIVO'] = ''

# Condición 1
condicion = df_no_validos_presencial['ESTADO_PAGO'] == 'NO PAGO'
df_no_validos_presencial.loc[condicion, 'MOTIVO'] += 'NO REGISTRA PAGO | '

# Condición 2
condicion = (
    (df_no_validos_presencial['NUEVO'] != 'NUEVO')
)
df_no_validos_presencial.loc[condicion, 'MOTIVO'] += 'NO ES UN ESTUDIANTE NUEVO | '

# Condición 3
condicion = df_no_validos_presencial['FuerzaComercialFinal'] != 'VINCULACIONES'
df_no_validos_presencial.loc[condicion, 'MOTIVO'] += 'PERTENECE A OTRA FUERZA | '

# Condición 4
condicion = df_no_validos_presencial['EST_MATRICULADO'] != '1-ACTIVO'
df_no_validos_presencial.loc[condicion, 'MOTIVO'] += 'MATRICULA INACTIVA | '

# Quitar el último " | "
df_no_validos_presencial['MOTIVO'] = (
    df_no_validos_presencial['MOTIVO']
    .str.rstrip(' |')
)

# Los que no tengan ningún motivo quedan como N/A
df_no_validos_presencial.loc[
    df_no_validos_presencial['MOTIVO'] == '',
    'MOTIVO'
] = 'N/A'

df_no_validos_presencial

,DOC_ALUM,PERIODO,ESTADO_PAGO,FuerzaComercialFinal,NUEVO,CLASE_ACTUAL,TIPO,PROGRAMA,EST_MATRICULADO,LLAVE_RECIBOS,Propietario del registro,Modalidad,Correo Ejecutivo,Regional,Documento Ejecutivo,Llave,MOTIVO
96,1072098738,2026D,PAGO,CONTACT,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS DE PROGRAMACIÓ...,1-ACTIVO,10720987382026D,Vinculaciones Nicolas Viafara Ospina,Presencial,NICOLAS_VIAFARAOS@CUN.EDU.CO,Ibague,1112496766,10720987382026D,PERTENECE A OTRA FUERZA
129,1082923026,2026D,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN OPERACIONES DEL PROCESO...,1-ACTIVO,10829230262026D,Vinculaciones Juan David Perez Perez,Presencial,JUAN_PEREZPE@CUN.EDU.CO,Santa Marta,1082917233,10829230262026D,NO REGISTRA PAGO
220,4382768,2026D,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN OPERACIONES DEL PROCESO...,1-ACTIVO,43827682026D,Vinculaciones Miguel Angel Palomino Peralta,Presencial,MIGUEL_PALOMINO@CUN.EDU.CO,Montería,1128053909,43827682026D,NO REGISTRA PAGO


In [367]:
# se traen las ventas que no tienen marca de pago
df_no_recibos_presencial = df_ventas_presencial[~df_ventas_presencial['Llave'].isin(df_recibos_presencial['LLAVE_RECIBOS'])].copy()
df_no_recibos_presencial['DOC_ALUM'] = df_no_recibos_presencial['Número de Documento de estudiante'].astype(str)
df_no_recibos_presencial['PERIODO'] = df_no_recibos_presencial['Periodo']
df_no_recibos_presencial['ESTADO_PAGO'] = ' '
df_no_recibos_presencial['EST_MATRICULADO'] = ' '
df_no_recibos_presencial['FuerzaComercialFinal'] = ' '
df_no_recibos_presencial['NUEVO'] = ' '
df_no_recibos_presencial['CLASE_ACTUAL'] = ' '
df_no_recibos_presencial['TIPO'] = ' '
df_no_recibos_presencial['PROGRAMA'] = df_no_recibos_presencial['Programa vendido'] 
df_no_recibos_presencial['LLAVE_RECIBOS'] = ' '
df_no_recibos_presencial['MOTIVO'] = 'NO CUENTA CON MARCA DE PAGO'
df_no_recibos_presencial = df_no_recibos_presencial[['DOC_ALUM', 'PERIODO', 'ESTADO_PAGO', 'FuerzaComercialFinal', 'NUEVO',
       'CLASE_ACTUAL', 'TIPO', 'PROGRAMA', 'EST_MATRICULADO', 'LLAVE_RECIBOS',
       'Propietario del registro', 'Modalidad', 'Correo Ejecutivo', 'Regional',
       'Documento Ejecutivo', 'Llave','MOTIVO']]
df_no_recibos_presencial.shape

(50, 17)

In [368]:
# SE UNIFICAN LOS NO VALIDOS Y LOS QUE NO TIENEN MARCA DE PAGO
df_no_validos_presencial_2 = pd.concat([df_no_validos_presencial,df_no_recibos_presencial])
df_no_validos_presencial_2

,DOC_ALUM,PERIODO,ESTADO_PAGO,FuerzaComercialFinal,NUEVO,CLASE_ACTUAL,TIPO,PROGRAMA,EST_MATRICULADO,LLAVE_RECIBOS,Propietario del registro,Modalidad,Correo Ejecutivo,Regional,Documento Ejecutivo,Llave,MOTIVO
96,1072098738,2026D,PAGO,CONTACT,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS DE PROGRAMACIÓ...,1-ACTIVO,10720987382026D,Vinculaciones Nicolas Viafara Ospina,Presencial,NICOLAS_VIAFARAOS@CUN.EDU.CO,Ibague,1112496766,10720987382026D,PERTENECE A OTRA FUERZA
129,1082923026,2026D,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN OPERACIONES DEL PROCESO...,1-ACTIVO,10829230262026D,Vinculaciones Juan David Perez Perez,Presencial,JUAN_PEREZPE@CUN.EDU.CO,Santa Marta,1082917233,10829230262026D,NO REGISTRA PAGO
220,4382768,2026D,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN OPERACIONES DEL PROCESO...,1-ACTIVO,43827682026D,Vinculaciones Miguel Angel Palomino Peralta,Presencial,MIGUEL_PALOMINO@CUN.EDU.CO,Montería,1128053909,43827682026D,NO REGISTRA PAGO
316,1067608339,2026D,,,,,,NEGOCIOS INTERNACIONALES,,,Vinculaciones Greys Paola Navarro Alvarado,Presencial,GREYS_NAVARROAL@CUN.EDU.CO,Santa Marta,1082905008,10676083392026D,NO CUENTA CON MARCA DE PAGO
465,1082837126,2026D,,,,,,ADMINISTRACION DE EMPRESAS,,,Vinculaciones Greys Paola Navarro Alvarado,Presencial,GREYS_NAVARROAL@CUN.EDU.CO,Santa Marta,1082905008,10828371262026D,NO CUENTA CON MARCA DE PAGO
898,1082916551,2026D,,,,,,DISENO DE MODAS,,,Vinculaciones Pedro Nel Rangel Luna,Presencial,PEDRO_RANGELLU@CUN.EDU.CO,Santa Marta,91112807,10829165512026D,NO CUENTA CON MARCA DE PAGO
915,1082980360,2026D,,,,,,ADMINISTRACION DE EMPRESAS,,,Vinculaciones Pedro Nel Rangel Luna,Presencial,PEDRO_RANGELLU@CUN.EDU.CO,Santa Marta,91112807,10829803602026D,NO CUENTA CON MARCA DE PAGO
2066,1082919023,2026D,,,,,,MARKETING DIGITAL,,,Vinculaciones Pedro Nel Rangel Luna,Presencial,PEDRO_RANGELLU@CUN.EDU.CO,Santa Marta,91112807,10829190232026D,NO CUENTA CON MARCA DE PAGO
2114,1101459825,2026D,,,,,,ADMINISTRACION DE EMPRESAS,,,Vinculaciones Fabio Andres Miranda Sequea,Presencial,FABIO_MIRANDA@CUN.EDU.CO,Sincelejo,1102877864,11014598252026D,NO CUENTA CON MARCA DE PAGO
2401,1004370350,2026D,,,,,,ADMINISTRACIÓN DE LA SEGURIDAD Y SALUD EN EL T...,,,Vinculaciones Pedro Nel Rangel Luna,Presencial,PEDRO_RANGELLU@CUN.EDU.CO,Santa Marta,91112807,10043703502026D,NO CUENTA CON MARCA DE PAGO


SE CRUZA CON OPORTUNIDADES

In [369]:
# se crea la llave para cruzar por doc alum+periodo+correo ejecutivo
df_validos_presencial['LLAVE_ZOHO'] = df_validos_presencial['DOC_ALUM']+df_validos_presencial['PERIODO']+df_validos_presencial['Correo Ejecutivo']

In [370]:
df_zoho_recibos_presencial = df_validos_presencial.merge(df_zoho_vinculaciones[['Propietario_de_Oportunidad_Name','Fase','Hora_de_modificación','LLAVE_ZOHO']],left_on='LLAVE_ZOHO',right_on='LLAVE_ZOHO',how='inner').copy()
df_zoho_recibos_presencial.drop_duplicates(inplace=True)
df_zoho_recibos_presencial.shape

(221, 20)

In [371]:
df_no_validos_zoho_presencial = df_validos_presencial[~df_validos_presencial['LLAVE_ZOHO'].isin(df_zoho_vinculaciones['LLAVE_ZOHO'])].copy()
df_no_validos_zoho_presencial['MOTIVO'] = 'NO ESTA OK EN ZOHO'
df_no_validos_zoho_presencial = df_no_validos_zoho_presencial.drop(columns=['LLAVE_ZOHO'])
df_no_validos_zoho_presencial.shape

(12, 17)

In [372]:
# SE UNIFICAN LOS NO VALIDOS TOTALES DE VIRTUALES
df_no_validos_presencial_3 = pd.concat([df_no_validos_presencial_2,df_no_validos_zoho_presencial])
df_no_validos_presencial_3

,DOC_ALUM,PERIODO,ESTADO_PAGO,FuerzaComercialFinal,NUEVO,CLASE_ACTUAL,TIPO,PROGRAMA,EST_MATRICULADO,LLAVE_RECIBOS,Propietario del registro,Modalidad,Correo Ejecutivo,Regional,Documento Ejecutivo,Llave,MOTIVO
96,1072098738,2026D,PAGO,CONTACT,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS DE PROGRAMACIÓ...,1-ACTIVO,10720987382026D,Vinculaciones Nicolas Viafara Ospina,Presencial,NICOLAS_VIAFARAOS@CUN.EDU.CO,Ibague,1112496766,10720987382026D,PERTENECE A OTRA FUERZA
129,1082923026,2026D,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN OPERACIONES DEL PROCESO...,1-ACTIVO,10829230262026D,Vinculaciones Juan David Perez Perez,Presencial,JUAN_PEREZPE@CUN.EDU.CO,Santa Marta,1082917233,10829230262026D,NO REGISTRA PAGO
220,4382768,2026D,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN OPERACIONES DEL PROCESO...,1-ACTIVO,43827682026D,Vinculaciones Miguel Angel Palomino Peralta,Presencial,MIGUEL_PALOMINO@CUN.EDU.CO,Montería,1128053909,43827682026D,NO REGISTRA PAGO
316,1067608339,2026D,,,,,,NEGOCIOS INTERNACIONALES,,,Vinculaciones Greys Paola Navarro Alvarado,Presencial,GREYS_NAVARROAL@CUN.EDU.CO,Santa Marta,1082905008,10676083392026D,NO CUENTA CON MARCA DE PAGO
465,1082837126,2026D,,,,,,ADMINISTRACION DE EMPRESAS,,,Vinculaciones Greys Paola Navarro Alvarado,Presencial,GREYS_NAVARROAL@CUN.EDU.CO,Santa Marta,1082905008,10828371262026D,NO CUENTA CON MARCA DE PAGO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,1062958813,2026D,PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN OPERACIONES DEL PROCESO...,1-ACTIVO,10629588132026D,Vinculaciones Miguel Angel Palomino Peralta,Presencial,MIGUEL_PALOMINO@CUN.EDU.CO,Montería,1128053909,10629588132026D,NO ESTA OK EN ZOHO
82,1062074474,2026D,PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN OPERACIÓN DE PROCESOS D...,1-ACTIVO,10620744742026D,Vinculaciones Jose Andres Navarro Ramirez,Presencial,JOSE_NAVARRORA@CUN.EDU.CO,Neiva,1075230146,10620744742026D,NO ESTA OK EN ZOHO
89,1067934169,2026D,PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN SOPORTE DE SISTEMAS E I...,1-ACTIVO,10679341692026D,Vinculaciones Miguel Angel Palomino Peralta,Presencial,MIGUEL_PALOMINO@CUN.EDU.CO,Montería,1128053909,10679341692026D,NO ESTA OK EN ZOHO
165,1104934835,2026D,PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS DE PROGRAMACIÓ...,1-ACTIVO,11049348352026D,Vinculaciones Karen Danitza Rosell Leyton,Presencial,KAREN_ROSELL@CUN.EDU.CO,Ibague,1110479583,11049348352026D,NO ESTA OK EN ZOHO


SE CRUZA CON EL HISTORICO

In [373]:
df_validos_presenciales_final = df_zoho_recibos_presencial[~df_zoho_recibos_presencial['Llave'].isin(df_historico['LLAVE_HISTORICO'])].copy()
df_validos_presenciales_final.shape

(221, 20)

In [374]:
df_no_validos_historico_presenciales = df_zoho_recibos_presencial[df_zoho_recibos_presencial['Llave'].isin(df_historico['LLAVE_HISTORICO'])].copy()
df_no_validos_historico_presenciales['MOTIVO'] = 'YA SE PAGO EN LA LIQUIDACION PASADA' 
df_no_validos_historico_presenciales = df_no_validos_historico_presenciales.drop(columns=['LLAVE_ZOHO','Propietario_de_Oportunidad_Name','Fase','Hora_de_modificación'])
df_no_validos_historico_presenciales.shape

(0, 17)

SE UNE TODOS LOS NO VALIDOS PRESENCIAL

In [375]:
df_novalidos_presencial_final = pd.concat([df_no_validos_presencial_3,df_no_validos_historico_presenciales])
df_novalidos_presencial_final.shape

(65, 17)

CALCULOS

In [376]:
# Filtrar y contar las ventas por estado de pago (PAGO y NO PAGO)
ventas_asesor_presenciales = (
    df_validos_presenciales_final
    .groupby(["Documento Ejecutivo"])
    .size()
    .reset_index(name="VENTAS_PRESENCIALES")
).copy()

In [377]:
df_metas_presencial['GESTOR DOCUMENTO'] = df_metas_presencial['GESTOR DOCUMENTO'].astype(str)
ventas_asesor_presenciales['Documento Ejecutivo'] = ventas_asesor_presenciales['Documento Ejecutivo'].astype(str)
ventas_asesor_presenciales = ventas_asesor_presenciales.merge(df_metas_presencial,how='left',left_on='Documento Ejecutivo',right_on='GESTOR DOCUMENTO').copy()
ventas_asesor_presenciales = ventas_asesor_presenciales.drop(columns=['GESTOR DOCUMENTO'])
ventas_asesor_presenciales

,Documento Ejecutivo,VENTAS_PRESENCIALES,GESTOR CORREO,CARGO,Meta,LIDER,SEDE,LOCALIDAD
0,1033377195,2,LAURA_BEDOYAC@CUN.EDU.CO,Gestor de ventas Junior,17,SANDRA,MEDELLIN,MEDELLIN
1,1053787428,22,DAVID_QUICENO@CUN.EDU.CO,Gestor de ventas Senior,16,SANDRA,MEDELLIN,MEDELLIN
2,1067283629,1,VICTOR_TOBON@CUN.EDU.CO,Gestor de ventas Junior,18,SANDRA,MONTERIA,MONTERIA
3,1081796706,21,JOEL_BOLANO@CUN.EDU.CO,Gestor de ventas Junior,17,SANDRA,BARRANQUILLA,BARRANQUILLA
4,1081811110,19,JESUS_JIMENEZ@CUN.EDU.CO,Gestor de ventas Junior,17,SANDRA,BARRANQUILLA,BARRANQUILLA
5,1082905008,4,GREYS_NAVARROAL@CUN.EDU.CO,Gestor de ventas Junior,18,SANDRA,SANTA MARTA,SANTA MARTA
6,1082917233,7,JUAN_PEREZPE@CUN.EDU.CO,Gestor de ventas Junior,18,SANDRA,SANTA MARTA,SANTA MARTA
7,1083015951,19,MANUEL_GARCIAP@CUN.EDU.CO,Gestor de ventas Senior,16,SANDRA,SANTA MARTA,SANTA MARTA
8,1102836456,2,JAIRO_VILLADIEGO@CUN.EDU.CO,Gestor de ventas Junior,18,SANDRA,SINCELEJO,SINCELEJO
9,1102843765,3,ARMANDO_SIERRAMA@CUN.EDU.CO,Gestor de ventas Junior,18,SANDRA,SINCELEJO,SINCELEJO


In [378]:
esquema_comisiones_presenciales = {

    "SANDRA": {
        "BARRANQUILLA": {
            "Gestor de ventas Senior":{ 
                        16: [  
                                (16, 25, 90000),
                                (26, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        17: [  
                                (17, 34, 90000),
                                (31, float('inf'), 120000)
                            
                            ]       
            }},

        "CARTAGENA": {
            "Gestor de ventas Senior":{ 
                        19: [  
                                (19, 30, 100000),
                                (31, float('inf'), 130000)
                            ]},
            "Gestor de ventas Junior":{ 
                        16: [  
                                (16, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ]       
            },
            "Gestor de ventas Nuevo":{ 
                        1: [  
                                (1, 5, 23000),
                                (5, 7, 100000),
                                (8, float('inf'), 130000)
                            
                            ]       
            }},
        "IBAGUE": {
            "Gestor de ventas Senior":{ 
                        16: [  
                                (16, 25, 90000),
                                (26, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        18: [  
                                (18, 34, 90000),
                                (35, float('inf'), 120000)
                            
                            ]       
            },
            "Gestor de ventas Nuevo":{ 
                        1: [  
                                (1, 18, 23000),
                                (18, 35, 90000),
                                (35, float('inf'), 120000)
                            
                            ]       
            }},
        "MEDELLIN": {
            "Gestor de ventas Senior":{ 
                        16: [  
                                (16, 25, 90000),
                                (26, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        17: [  
                                (17, 34, 90000),
                                (35, float('inf'), 120000)
                            
                            ]       
            }},
        "MONTERIA": {
            "Gestor de ventas Senior":{ 
                        16: [  
                                (16, 25, 90000),
                                (26, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        18: [  
                                (18, 34, 90000),
                                (35, float('inf'), 120000)
                            
                            ]       
            }},
        "NEIVA": {
            "Gestor de ventas Senior":{ 
                        16: [  
                                (16, 25, 90000),
                                (26, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        18: [  
                                (18, 34, 90000),
                                (35, float('inf'), 120000)
                            
                            ]       
            },
            "Gestor de ventas Nuevo":{ 
                        1: [  
                                (1, 3, 23000),
                                (3, 5, 90000),
                                (6, float('inf'), 120000)
                            
                            ]       
            }},
        "SANTA MARTA": {
            "Gestor de ventas Senior":{ 
                        16: [  
                                (16, 25, 90000),
                                (26, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        18: [  
                                (18, 34, 90000),
                                (35, float('inf'), 120000)
                            
                            ]       
            }},
        "SINCELEJO": {
            "Gestor de ventas Senior":{ 
                        16: [  
                                (16, 25, 90000),
                                (26, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        18: [  
                                (18, 34, 90000),
                                (35, float('inf'), 120000)
                            
                            ]       
            }},               
        "CALI": {
            "Gestor de ventas Senior":{ 
                        19: [  
                                (19, 30, 100000),
                                (31, float('inf'), 130000)
                            ]},
            "Gestor de ventas Junior":{ 
                        21: [  
                                (21, 30, 100000),
                                (31, float('inf'), 130000)],
                        16: [  
                                (16, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ]            
            }}, 
        "PEREIRA": {
            "Gestor de ventas Senior":{ 
                        19: [  
                                (19, 30, 100000),
                                (31, float('inf'), 130000)
                            ]},
            "Gestor de ventas Junior":{ 
                        21: [  
                                (21, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ],
                        16: [  
                                (16, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ]     
                            }},
        "BUCARAMANGA": {
            "Gestor de ventas Senior":{ 
                        19: [  
                                (19, 30, 100000),
                                (31, float('inf'), 130000)
                            ]},
            "Gestor de ventas Junior":{ 
                        21: [  
                                (21, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ],
                        16: [  
                                (16, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ]     }},
         "TUNJA": {
            "Gestor de ventas Senior":{ 
                        17: [  
                                (17, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ]  },
            "Gestor de ventas Junior":{ 
                        21: [  
                                (21, 30, 100000),
                                (31, float('inf'), 130000)
                            
                            ]       
            }}            
    }}

In [379]:
def asignar_comision_presenciales(lider, sede, cargo, meta, ventas):

    # 1️⃣ Validar líder
    if lider not in esquema_comisiones_presenciales:
        return 0

    # 2️⃣ Validar sede
    if sede not in esquema_comisiones_presenciales[lider]:
        return 0

    # 3️⃣ Validar cargo
    if cargo not in esquema_comisiones_presenciales[lider][sede]:
        return 0

    # 4️⃣ Validar meta
    if meta not in esquema_comisiones_presenciales[lider][sede][cargo]:
        return 0

    # 5️⃣ Validar cumplimiento de meta
    if ventas < meta:
        return 0

    # 6️⃣ Obtener los pisos correspondientes a la meta
    rangos = esquema_comisiones_presenciales[lider][sede][cargo][meta]

    # 7️⃣ Buscar el piso correspondiente según las ventas
    for min_rango, max_rango, comision in rangos:

        if min_rango <= ventas <= max_rango:

            # Comisión por venta × número de ventas
            return ventas * comision

    return 0

In [380]:
ventas_asesor_presenciales["COMISION_TOTAL_PRESENCIAL"] = ventas_asesor_presenciales.apply(
    lambda row: asignar_comision_presenciales(
        row["LIDER"],
        row["SEDE"],
        row["CARGO"],
        row["Meta"],
        row["VENTAS_PRESENCIALES"]
    ),
    axis=1
)
ventas_asesor_presenciales['COMISION_UNITARIA_PRESENCIAL'] = ventas_asesor_presenciales['COMISION_TOTAL_PRESENCIAL'] / ventas_asesor_presenciales['VENTAS_PRESENCIALES']
ventas_asesor_presenciales['%CUMPLIMIENTO_PRESENCIAL'] = (
    (ventas_asesor_presenciales['VENTAS_PRESENCIALES'] / ventas_asesor_presenciales['Meta']) * 100).round(2).astype(str) + '%'
ventas_asesor_presenciales

,Documento Ejecutivo,VENTAS_PRESENCIALES,GESTOR CORREO,CARGO,Meta,LIDER,SEDE,LOCALIDAD,COMISION_TOTAL_PRESENCIAL,COMISION_UNITARIA_PRESENCIAL,%CUMPLIMIENTO_PRESENCIAL
0,1033377195,2,LAURA_BEDOYAC@CUN.EDU.CO,Gestor de ventas Junior,17,SANDRA,MEDELLIN,MEDELLIN,0,0.0,11.76%
1,1053787428,22,DAVID_QUICENO@CUN.EDU.CO,Gestor de ventas Senior,16,SANDRA,MEDELLIN,MEDELLIN,1980000,90000.0,137.5%
2,1067283629,1,VICTOR_TOBON@CUN.EDU.CO,Gestor de ventas Junior,18,SANDRA,MONTERIA,MONTERIA,0,0.0,5.56%
3,1081796706,21,JOEL_BOLANO@CUN.EDU.CO,Gestor de ventas Junior,17,SANDRA,BARRANQUILLA,BARRANQUILLA,1890000,90000.0,123.53%
4,1081811110,19,JESUS_JIMENEZ@CUN.EDU.CO,Gestor de ventas Junior,17,SANDRA,BARRANQUILLA,BARRANQUILLA,1710000,90000.0,111.76%
5,1082905008,4,GREYS_NAVARROAL@CUN.EDU.CO,Gestor de ventas Junior,18,SANDRA,SANTA MARTA,SANTA MARTA,0,0.0,22.22%
6,1082917233,7,JUAN_PEREZPE@CUN.EDU.CO,Gestor de ventas Junior,18,SANDRA,SANTA MARTA,SANTA MARTA,0,0.0,38.89%
7,1083015951,19,MANUEL_GARCIAP@CUN.EDU.CO,Gestor de ventas Senior,16,SANDRA,SANTA MARTA,SANTA MARTA,1710000,90000.0,118.75%
8,1102836456,2,JAIRO_VILLADIEGO@CUN.EDU.CO,Gestor de ventas Junior,18,SANDRA,SINCELEJO,SINCELEJO,0,0.0,11.11%
9,1102843765,3,ARMANDO_SIERRAMA@CUN.EDU.CO,Gestor de ventas Junior,18,SANDRA,SINCELEJO,SINCELEJO,0,0.0,16.67%


# SE LIQUIDAN ESPECIALIZACIONES

In [381]:
df_metas_especializacion = pd.read_excel(r'C:\Users\alejandro_arboledas\OneDrive - Corporación Unificada Nacional de Educación Superior - CUN\Documentos\PYTHON\VINCULACIONES\MODELO3\2026\METAS_202609.xlsx', sheet_name= periodo_especializacion)
df_metas_especializacion

,GESTOR DOCUMENTO,GESTOR CORREO,CARGO,Meta,LIDER,SEDE,LOCALIDAD
0,1143244777,katherine_bohorquez@cun.edu.co,Gestor de ventas Senior,8,SANDRA,BARRANQUILLA,BARRANQUILLA
1,1081796706,joel_bolano@cun.edu.co,Gestor de ventas Junior,12,SANDRA,BARRANQUILLA,BARRANQUILLA
2,1081811110,jesus_jimenez@cun.edu.co,Gestor de ventas Junior,12,SANDRA,BARRANQUILLA,BARRANQUILLA
3,44156610,claudia_guzmanbe@cun.edu.co,Gestor de ventas Junior,12,SANDRA,BARRANQUILLA,BARRANQUILLA
4,45550304,katherine_rosado@cun.edu.co,Gestor de ventas Senior,13,SANDRA,CARTAGENA,CARTAGENA
5,1235040095,rosa_ahumada@cun.edu.co,Gestor de ventas Nuevo,1,SANDRA,CARTAGENA,CARTAGENA
6,1143329915,ronald_villalbay@cun.edu.co,Gestor de ventas Junior,13,SANDRA,CARTAGENA,CARTAGENA
7,25800034,sandra_durango@cun.edu.co,Gestor de ventas Junior,13,SANDRA,CARTAGENA,CARTAGENA
8,1110479583,karen_rosell@cun.edu.co,Gestor de ventas Senior,8,SANDRA,IBAGUE,IBAGUE
9,1110587566,margarita_navarro@cun.edu.co,Gestor de ventas Junior,12,SANDRA,IBAGUE,IBAGUE


In [382]:
df_metas_especializacion = df_metas_especializacion[df_metas_especializacion['LIDER'] == lider]
df_metas_especializacion.shape

(48, 7)

In [383]:
df_metas_especializacion['GESTOR CORREO'] = df_metas_especializacion['GESTOR CORREO'].str.upper()
df_ventas_especializacion = df_pre_ventas[df_pre_ventas['Correo Ejecutivo'].isin(df_metas_especializacion['GESTOR CORREO'])].copy()
df_ventas_especializacion = df_ventas_especializacion[df_ventas_especializacion['Periodo'].isin([periodo_especializacion])]
df_ventas_especializacion.shape

(705, 32)

In [384]:
columnas = ['Hora agregado', 'Numero de recibo','Dirección IP','Correo del estudiante','Teléfono del estudiante',
'Ciudad de residencia','Imagen del Pago Generado','% De Descuento',
'Medio de pago','Fuente de pionero','Campana','Fuente campana','Content','Sitio web', 'Tipo de inscripción',
'ID único','Fecha','Documento Ejecutivo_x','Número de aprobación de pago','Tipo de Financiación', 'Convenio', 'Valor pagado']

df_ventas_especializacion.drop(columns=columnas, inplace=True)
df_ventas_especializacion['Periodo'] = df_ventas_especializacion['Periodo'].astype(str).str.strip()
df_ventas_especializacion = df_ventas_especializacion.rename(columns={'Documento Ejecutivo_y':'Documento Ejecutivo'})
df_ventas_especializacion['Documento Ejecutivo'] = df_ventas_especializacion['Documento Ejecutivo'].astype(str)
df_ventas_especializacion['Número de Documento de estudiante'] = df_ventas_especializacion['Número de Documento de estudiante'].astype(str)
df_ventas_especializacion['Llave']=df_ventas_especializacion['Número de Documento de estudiante']+df_ventas_especializacion['Periodo']
df_ventas_especializacion.shape

(705, 11)

SE CRUZA CON RECIBOS

In [385]:
df_recibos_especializacion = df_recibos[df_recibos['PERIODO'] == periodo_especializacion].copy()
df_recibos_especializacion.shape

(2103, 10)

In [386]:
df_recibos_especializacion = df_recibos_especializacion.merge(df_ventas_especializacion[['Propietario del registro','Modalidad','Correo Ejecutivo', 'Regional',
       'Documento Ejecutivo', 'Llave']],how='inner',left_on='LLAVE_RECIBOS',right_on='Llave')
df_recibos_especializacion.drop_duplicates(inplace=True)
df_recibos_especializacion.shape

(692, 16)

In [390]:
#tratamiento de columnas
df_recibos_especializacion['CLASE_ACTUAL'] = df_recibos_especializacion['CLASE_ACTUAL'].astype(str).str.upper()
df_recibos_especializacion['FuerzaComercialFinal'] = df_recibos_especializacion['FuerzaComercialFinal'].str.upper()
df_recibos_especializacion['NUEVO'] = df_recibos_especializacion['NUEVO'].astype(str).str.upper()
df_recibos_especializacion['ESTADO_PAGO'] = df_recibos_especializacion['ESTADO_PAGO'].astype(str).str.upper()
df_recibos_especializacion['FuerzaComercialFinal'] = df_recibos_especializacion['FuerzaComercialFinal'].str.strip()
df_recibos_especializacion['NUEVO'] = df_recibos_especializacion['NUEVO'].str.strip()
df_recibos_especializacion['CLASE_ACTUAL'] = df_recibos_especializacion['CLASE_ACTUAL'].str.strip()
df_recibos_especializacion['ESTADO_PAGO'] = df_recibos_especializacion['ESTADO_PAGO'].str.strip()
df_recibos_especializacion['EST_MATRICULADO'] = (
    df_recibos_especializacion['EST_MATRICULADO']
    .astype(str)
    .str.strip()
    .str.upper()
)

In [391]:
# se divide entre validos
df_validos_especializacion = df_recibos_especializacion[
    ((df_recibos_especializacion['FuerzaComercialFinal'] == 'BASICA Y MEDIA') | (df_recibos_especializacion['FuerzaComercialFinal'] == 'VINCULACIONES') | (df_recibos_especializacion['FuerzaComercialFinal'] == 'INDETERMINADO')) &
    (df_recibos_especializacion['NUEVO'] == 'NUEVO') &
    (df_recibos_especializacion['EST_MATRICULADO'] == '1-ACTIVO') & 
    (df_recibos_especializacion['ESTADO_PAGO'] == 'PAGO')
].copy()
df_validos_especializacion.drop_duplicates(inplace=True)
df_validos_especializacion.shape

(679, 16)

In [392]:
# se divide entre no validos
mask = (
    df_recibos_especializacion['FuerzaComercialFinal'].isin(
        ['BASICA Y MEDIA', 'VINCULACIONES', 'INDETERMINADO']
    ) &
    (df_recibos_especializacion['NUEVO'] == 'NUEVO') &
    (df_recibos_especializacion['EST_MATRICULADO'] == '1-ACTIVO') &
    (df_recibos_especializacion['ESTADO_PAGO'] == 'PAGO')
)
df_no_validos_especializacion = df_recibos_especializacion[~mask].copy()
df_no_validos_especializacion.drop_duplicates(inplace=True)
df_no_validos_especializacion.shape

(13, 16)

In [393]:
# Inicializar MOTIVO vacío
df_no_validos_especializacion['MOTIVO'] = ''

# Condición 1
condicion = df_no_validos_especializacion['ESTADO_PAGO'] == 'NO PAGO'
df_no_validos_especializacion.loc[condicion, 'MOTIVO'] += 'NO REGISTRA PAGO | '

# Condición 2
condicion = (
    (df_no_validos_especializacion['NUEVO'] != 'NUEVO')
)
df_no_validos_especializacion.loc[condicion, 'MOTIVO'] += 'NO ES UN ESTUDIANTE NUEVO | '

# Condición 3
condicion = df_no_validos_especializacion['FuerzaComercialFinal'] != 'VINCULACIONES'
df_no_validos_especializacion.loc[condicion, 'MOTIVO'] += 'PERTENECE A OTRA FUERZA | '

# Condición 4
condicion = df_no_validos_especializacion['EST_MATRICULADO'] != '1-ACTIVO'
df_no_validos_especializacion.loc[condicion, 'MOTIVO'] += 'MATRICULA INACTIVA | '

# Quitar el último " | "
df_no_validos_especializacion['MOTIVO'] = (
    df_no_validos_especializacion['MOTIVO']
    .str.rstrip(' |')
)

# Los que no tengan ningún motivo quedan como N/A
df_no_validos_especializacion.loc[
    df_no_validos_especializacion['MOTIVO'] == '',
    'MOTIVO'
] = 'N/A'

df_no_validos_especializacion

,DOC_ALUM,PERIODO,ESTADO_PAGO,FuerzaComercialFinal,NUEVO,CLASE_ACTUAL,TIPO,PROGRAMA,EST_MATRICULADO,LLAVE_RECIBOS,Propietario del registro,Modalidad,Correo Ejecutivo,Regional,Documento Ejecutivo,Llave,MOTIVO
80,1007403332,26ES5,PAGO,TELECAMPUS,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN GERENCIA DE PROYECTOS,1-ACTIVO,100740333226ES5,Vinculaciones David Alejandro Orjuela Chaparro,Virtual,DAVID_ORJUELAC@CUN.EDU.CO,Tunja,1025521082,100740333226ES5,PERTENECE A OTRA FUERZA
129,1023960066,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN GERENCIA DE PROYECTOS,1-ACTIVO,102396006626ES5,Vinculaciones Alexander Botia Alvarez,Virtual,ALEXANDER_BOTIA@CUN.EDU.CO,Bucaramanga,1098652444,102396006626ES5,NO REGISTRA PAGO
156,1037644266,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN DESARROLLO ORGANIZACIONAL Y...,1-ACTIVO,103764426626ES5,Vinculaciones Laura Yovana Bedoya Ceballos,Virtual,LAURA_BEDOYAC@CUN.EDU.CO,Medellin,1033377195,103764426626ES5,NO REGISTRA PAGO
220,1063726296,26ES5,PAGO,CONTACT,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN SEGURIDAD Y SALUD EN EL TRA...,1-ACTIVO,106372629626ES5,Vinculaciones Erick Mauricio Carmona Vergara,Virtual,ERICK_CARMONA@CUN.EDU.CO,Montería,1067944674,106372629626ES5,PERTENECE A OTRA FUERZA
328,1081824356,26ES5,PAGO,CONTACT,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN GERENCIA FINANCIERA,1-ACTIVO,108182435626ES5,Vinculaciones David Alejandro Orjuela Chaparro,Virtual,DAVID_ORJUELAC@CUN.EDU.CO,Tunja,1025521082,108182435626ES5,PERTENECE A OTRA FUERZA
426,1100340568,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN SEGURIDAD Y SALUD EN EL TRA...,1-ACTIVO,110034056826ES5,Vinculaciones Fabio Andres Miranda Sequea,Virtual,FABIO_MIRANDA@CUN.EDU.CO,Sincelejo,1102877864,110034056826ES5,NO REGISTRA PAGO
465,1117550518,26ES5,PAGO,PERMANENCIA,ANTIGUO,NUEVO,Postgrado,ESPECIALIZACIÓN EN MARKETING DIGITAL,1-ACTIVO,111755051826ES5,Vinculaciones Margarita Rosa Navarro Garzon,Virtual,MARGARITA_NAVARRO@CUN.EDU.CO,Ibague,1110587566,111755051826ES5,NO ES UN ESTUDIANTE NUEVO | PERTENECE A OTRA F...
496,1131519245,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN GERENCIA EDUCATIVA,1-ACTIVO,113151924526ES5,Vinculaciones David Alejandro Orjuela Chaparro,Virtual,DAVID_ORJUELAC@CUN.EDU.CO,Tunja,1025521082,113151924526ES5,NO REGISTRA PAGO
580,38888947,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN CONTRATACIÓN ESTATAL,1-ACTIVO,3888894726ES5,Vinculaciones Valentina Montoya Cruz,Virtual,VALENTINA_MONTOYAC@CUN.EDU.CO,Cali,1006053851,3888894726ES5,NO REGISTRA PAGO
588,51952090,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN GERENCIA EDUCATIVA,1-ACTIVO,5195209026ES5,Vinculaciones Nicolas Viafara Ospina,Virtual,NICOLAS_VIAFARAOS@CUN.EDU.CO,Ibague,1112496766,5195209026ES5,NO REGISTRA PAGO


In [394]:
# se traen las ventas que no tienen marca de pago
df_no_recibos_especializacion = df_ventas_especializacion[~df_ventas_especializacion['Llave'].isin(df_recibos_especializacion['LLAVE_RECIBOS'])].copy()
df_no_recibos_especializacion['DOC_ALUM'] = df_no_recibos_especializacion['Número de Documento de estudiante'].astype(str)
df_no_recibos_especializacion['PERIODO'] = df_no_recibos_especializacion['Periodo']
df_no_recibos_especializacion['ESTADO_PAGO'] = ' '
df_no_recibos_especializacion['EST_MATRICULADO'] = ' '
df_no_recibos_especializacion['FuerzaComercialFinal'] = ' '
df_no_recibos_especializacion['NUEVO'] = ' '
df_no_recibos_especializacion['CLASE_ACTUAL'] = ' '
df_no_recibos_especializacion['TIPO'] = ' '
df_no_recibos_especializacion['PROGRAMA'] = df_no_recibos_especializacion['Programa vendido'] 
df_no_recibos_especializacion['LLAVE_RECIBOS'] = ' '
df_no_recibos_especializacion['MOTIVO'] = 'NO CUENTA CON MARCA DE PAGO'
df_no_recibos_especializacion = df_no_recibos_especializacion[['DOC_ALUM', 'PERIODO', 'ESTADO_PAGO', 'FuerzaComercialFinal', 'NUEVO',
       'CLASE_ACTUAL', 'TIPO', 'PROGRAMA', 'EST_MATRICULADO', 'LLAVE_RECIBOS',
       'Propietario del registro', 'Modalidad', 'Correo Ejecutivo', 'Regional',
       'Documento Ejecutivo', 'Llave','MOTIVO']]
df_no_recibos_especializacion.shape

(10, 17)

In [395]:
# SE UNIFICAN LOS NO VALIDOS Y LOS QUE NO TIENEN MARCA DE PAGO
df_no_validos_especializacion_2 = pd.concat([df_no_validos_especializacion,df_no_recibos_especializacion])
df_no_validos_especializacion_2

,DOC_ALUM,PERIODO,ESTADO_PAGO,FuerzaComercialFinal,NUEVO,CLASE_ACTUAL,TIPO,PROGRAMA,EST_MATRICULADO,LLAVE_RECIBOS,Propietario del registro,Modalidad,Correo Ejecutivo,Regional,Documento Ejecutivo,Llave,MOTIVO
80,1007403332,26ES5,PAGO,TELECAMPUS,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN GERENCIA DE PROYECTOS,1-ACTIVO,100740333226ES5,Vinculaciones David Alejandro Orjuela Chaparro,Virtual,DAVID_ORJUELAC@CUN.EDU.CO,Tunja,1025521082,100740333226ES5,PERTENECE A OTRA FUERZA
129,1023960066,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN GERENCIA DE PROYECTOS,1-ACTIVO,102396006626ES5,Vinculaciones Alexander Botia Alvarez,Virtual,ALEXANDER_BOTIA@CUN.EDU.CO,Bucaramanga,1098652444,102396006626ES5,NO REGISTRA PAGO
156,1037644266,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN DESARROLLO ORGANIZACIONAL Y...,1-ACTIVO,103764426626ES5,Vinculaciones Laura Yovana Bedoya Ceballos,Virtual,LAURA_BEDOYAC@CUN.EDU.CO,Medellin,1033377195,103764426626ES5,NO REGISTRA PAGO
220,1063726296,26ES5,PAGO,CONTACT,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN SEGURIDAD Y SALUD EN EL TRA...,1-ACTIVO,106372629626ES5,Vinculaciones Erick Mauricio Carmona Vergara,Virtual,ERICK_CARMONA@CUN.EDU.CO,Montería,1067944674,106372629626ES5,PERTENECE A OTRA FUERZA
328,1081824356,26ES5,PAGO,CONTACT,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN GERENCIA FINANCIERA,1-ACTIVO,108182435626ES5,Vinculaciones David Alejandro Orjuela Chaparro,Virtual,DAVID_ORJUELAC@CUN.EDU.CO,Tunja,1025521082,108182435626ES5,PERTENECE A OTRA FUERZA
426,1100340568,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN SEGURIDAD Y SALUD EN EL TRA...,1-ACTIVO,110034056826ES5,Vinculaciones Fabio Andres Miranda Sequea,Virtual,FABIO_MIRANDA@CUN.EDU.CO,Sincelejo,1102877864,110034056826ES5,NO REGISTRA PAGO
465,1117550518,26ES5,PAGO,PERMANENCIA,ANTIGUO,NUEVO,Postgrado,ESPECIALIZACIÓN EN MARKETING DIGITAL,1-ACTIVO,111755051826ES5,Vinculaciones Margarita Rosa Navarro Garzon,Virtual,MARGARITA_NAVARRO@CUN.EDU.CO,Ibague,1110587566,111755051826ES5,NO ES UN ESTUDIANTE NUEVO | PERTENECE A OTRA F...
496,1131519245,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN GERENCIA EDUCATIVA,1-ACTIVO,113151924526ES5,Vinculaciones David Alejandro Orjuela Chaparro,Virtual,DAVID_ORJUELAC@CUN.EDU.CO,Tunja,1025521082,113151924526ES5,NO REGISTRA PAGO
580,38888947,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN CONTRATACIÓN ESTATAL,1-ACTIVO,3888894726ES5,Vinculaciones Valentina Montoya Cruz,Virtual,VALENTINA_MONTOYAC@CUN.EDU.CO,Cali,1006053851,3888894726ES5,NO REGISTRA PAGO
588,51952090,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN GERENCIA EDUCATIVA,1-ACTIVO,5195209026ES5,Vinculaciones Nicolas Viafara Ospina,Virtual,NICOLAS_VIAFARAOS@CUN.EDU.CO,Ibague,1112496766,5195209026ES5,NO REGISTRA PAGO


SE CRUZA CON OPORTUNIDADES

In [396]:
# se crea la llave para cruzar por doc alum+periodo+correo ejecutivo
df_validos_especializacion['LLAVE_ZOHO'] = df_validos_especializacion['DOC_ALUM']+df_validos_especializacion['PERIODO']+df_validos_especializacion['Correo Ejecutivo']

In [397]:
df_zoho_recibos_especializacion = df_validos_especializacion.merge(df_zoho_vinculaciones[['Propietario_de_Oportunidad_Name','Fase','Hora_de_modificación','LLAVE_ZOHO']],left_on='LLAVE_ZOHO',right_on='LLAVE_ZOHO',how='inner').copy()
df_zoho_recibos_especializacion.drop_duplicates(inplace=True)
df_zoho_recibos_especializacion.shape

(669, 20)

In [398]:
df_no_validos_zoho_especializacion = df_validos_especializacion[~df_validos_especializacion['LLAVE_ZOHO'].isin(df_zoho_vinculaciones['LLAVE_ZOHO'])].copy()
df_no_validos_zoho_especializacion['MOTIVO'] = 'NO ESTA OK EN ZOHO'
df_no_validos_zoho_especializacion = df_no_validos_zoho_especializacion.drop(columns=['LLAVE_ZOHO'])
df_no_validos_zoho_especializacion.shape

(14, 17)

In [399]:
# SE UNIFICAN LOS NO VALIDOS TOTALES DE VIRTUALES
df_no_validos_especializacion_3 = pd.concat([df_no_validos_especializacion_2,df_no_validos_zoho_especializacion])
df_no_validos_especializacion_3

,DOC_ALUM,PERIODO,ESTADO_PAGO,FuerzaComercialFinal,NUEVO,CLASE_ACTUAL,TIPO,PROGRAMA,EST_MATRICULADO,LLAVE_RECIBOS,Propietario del registro,Modalidad,Correo Ejecutivo,Regional,Documento Ejecutivo,Llave,MOTIVO
80,1007403332,26ES5,PAGO,TELECAMPUS,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN GERENCIA DE PROYECTOS,1-ACTIVO,100740333226ES5,Vinculaciones David Alejandro Orjuela Chaparro,Virtual,DAVID_ORJUELAC@CUN.EDU.CO,Tunja,1025521082,100740333226ES5,PERTENECE A OTRA FUERZA
129,1023960066,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN GERENCIA DE PROYECTOS,1-ACTIVO,102396006626ES5,Vinculaciones Alexander Botia Alvarez,Virtual,ALEXANDER_BOTIA@CUN.EDU.CO,Bucaramanga,1098652444,102396006626ES5,NO REGISTRA PAGO
156,1037644266,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN DESARROLLO ORGANIZACIONAL Y...,1-ACTIVO,103764426626ES5,Vinculaciones Laura Yovana Bedoya Ceballos,Virtual,LAURA_BEDOYAC@CUN.EDU.CO,Medellin,1033377195,103764426626ES5,NO REGISTRA PAGO
220,1063726296,26ES5,PAGO,CONTACT,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN SEGURIDAD Y SALUD EN EL TRA...,1-ACTIVO,106372629626ES5,Vinculaciones Erick Mauricio Carmona Vergara,Virtual,ERICK_CARMONA@CUN.EDU.CO,Montería,1067944674,106372629626ES5,PERTENECE A OTRA FUERZA
328,1081824356,26ES5,PAGO,CONTACT,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN GERENCIA FINANCIERA,1-ACTIVO,108182435626ES5,Vinculaciones David Alejandro Orjuela Chaparro,Virtual,DAVID_ORJUELAC@CUN.EDU.CO,Tunja,1025521082,108182435626ES5,PERTENECE A OTRA FUERZA
426,1100340568,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN SEGURIDAD Y SALUD EN EL TRA...,1-ACTIVO,110034056826ES5,Vinculaciones Fabio Andres Miranda Sequea,Virtual,FABIO_MIRANDA@CUN.EDU.CO,Sincelejo,1102877864,110034056826ES5,NO REGISTRA PAGO
465,1117550518,26ES5,PAGO,PERMANENCIA,ANTIGUO,NUEVO,Postgrado,ESPECIALIZACIÓN EN MARKETING DIGITAL,1-ACTIVO,111755051826ES5,Vinculaciones Margarita Rosa Navarro Garzon,Virtual,MARGARITA_NAVARRO@CUN.EDU.CO,Ibague,1110587566,111755051826ES5,NO ES UN ESTUDIANTE NUEVO | PERTENECE A OTRA F...
496,1131519245,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN GERENCIA EDUCATIVA,1-ACTIVO,113151924526ES5,Vinculaciones David Alejandro Orjuela Chaparro,Virtual,DAVID_ORJUELAC@CUN.EDU.CO,Tunja,1025521082,113151924526ES5,NO REGISTRA PAGO
580,38888947,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN CONTRATACIÓN ESTATAL,1-ACTIVO,3888894726ES5,Vinculaciones Valentina Montoya Cruz,Virtual,VALENTINA_MONTOYAC@CUN.EDU.CO,Cali,1006053851,3888894726ES5,NO REGISTRA PAGO
588,51952090,26ES5,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN GERENCIA EDUCATIVA,1-ACTIVO,5195209026ES5,Vinculaciones Nicolas Viafara Ospina,Virtual,NICOLAS_VIAFARAOS@CUN.EDU.CO,Ibague,1112496766,5195209026ES5,NO REGISTRA PAGO


SE CRUZA CON EL HISTORICO

In [400]:
df_validos_especializaciones_final = df_zoho_recibos_especializacion[~df_zoho_recibos_especializacion['Llave'].isin(df_historico['LLAVE_HISTORICO'])].copy()
df_validos_especializaciones_final.shape

(669, 20)

In [401]:
df_no_validos_historico_especializaciones = df_zoho_recibos_presencial[df_zoho_recibos_presencial['Llave'].isin(df_historico['LLAVE_HISTORICO'])].copy()
df_no_validos_historico_especializaciones['MOTIVO'] = 'YA SE PAGO EN LA LIQUIDACION PASADA' 
df_no_validos_historico_especializaciones = df_no_validos_historico_especializaciones.drop(columns=['LLAVE_ZOHO','Propietario_de_Oportunidad_Name','Fase','Hora_de_modificación'])
df_no_validos_historico_especializaciones.shape

(0, 17)

SE UNIFICAN LOS NO VALIDOS ESPECIALIZACION

In [402]:
df_novalidos_especializacion_final = pd.concat([df_no_validos_especializacion_3,df_no_validos_historico_especializaciones])
df_novalidos_especializacion_final.shape

(37, 17)

SE INICIAN CALCULOS DE COMISION

In [403]:
# Filtrar y contar las ventas por estado de pago (PAGO y NO PAGO)
ventas_asesor_especializaciones = (
    df_validos_especializaciones_final
    .groupby(["Documento Ejecutivo"])
    .size()
    .reset_index(name="VENTAS_ESPECIALIZACION")
).copy()

In [404]:
df_metas_especializacion['GESTOR DOCUMENTO'] = df_metas_especializacion['GESTOR DOCUMENTO'].astype(str)
ventas_asesor_especializaciones['Documento Ejecutivo'] = ventas_asesor_especializaciones['Documento Ejecutivo'].astype(str)
ventas_asesor_especializaciones = ventas_asesor_especializaciones.merge(df_metas_especializacion,how='left',left_on='Documento Ejecutivo',right_on='GESTOR DOCUMENTO').copy()
ventas_asesor_especializaciones = ventas_asesor_especializaciones.drop(columns=['GESTOR DOCUMENTO'])
ventas_asesor_especializaciones

,Documento Ejecutivo,VENTAS_ESPECIALIZACION,GESTOR CORREO,CARGO,Meta,LIDER,SEDE,LOCALIDAD
0,1000224303,1,GUISELLE_VALENZUELA@CUN.EDU.CO,Gestor de ventas Nuevo,1,SANDRA,IBAGUE,IBAGUE
1,1005337691,2,KAREN_AGUILAR@CUN.EDU.CO,Gestor de ventas Junior,15,SANDRA,BUCARAMANGA,BUCARAMANGA
2,1006053851,23,VALENTINA_MONTOYAC@CUN.EDU.CO,Gestor de ventas Junior,15,SANDRA,CALI,CALI
3,1023938622,1,LUIS_MUNOZMU@CUN.EDU.CO,Gestor de ventas Junior,15,SANDRA,PEREIRA,PEREIRA
4,1025521082,18,DAVID_ORJUELAC@CUN.EDU.CO,Gestor de ventas Junior,15,SANDRA,TUNJA,TUNJA
5,1033377195,27,LAURA_BEDOYAC@CUN.EDU.CO,Gestor de ventas Junior,12,SANDRA,MEDELLIN,MEDELLIN
6,1049637832,17,NATHALIA_FUQUENEY@CUN.EDU.CO,Gestor de ventas Junior,15,SANDRA,TUNJA,TUNJA
7,1053616791,23,MIRYAM_CALLEJASP@CUN.EDU.CO,Gestor de ventas Junior,15,SANDRA,TUNJA,TUNJA
8,1053787428,1,DAVID_QUICENO@CUN.EDU.CO,Gestor de ventas Senior,8,SANDRA,MEDELLIN,MEDELLIN
9,1067283629,17,VICTOR_TOBON@CUN.EDU.CO,Gestor de ventas Junior,12,SANDRA,MONTERIA,MONTERIA


In [405]:
esquema_comisiones_especializaciones = {

    "SANDRA": {
        "BARRANQUILLA": {
            "Gestor de ventas Senior":{ 
                        8: [  
                                (8, 20, 90000),
                                (21, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        12: [  
                                (12, 19, 90000),
                                (20, float('inf'), 120000)
                            
                            ]       
            }},

        "CARTAGENA": {
            "Gestor de ventas Senior":{ 
                        13: [  
                                (13, 19, 100000),
                                (20, float('inf'), 130000)
                            ]},
            "Gestor de ventas Junior":{ 
                        13: [  
                                (13, 19, 100000),
                                (20, float('inf'), 130000)
                            
                            ]       
            },
            "Gestor de ventas Nuevo":{ 
                        1: [  
                                (1, 5, 23000),
                                (5, 7, 100000),
                                (8, float('inf'), 130000)
                            
                            ]       
            }},
        "IBAGUE": {
            "Gestor de ventas Senior":{ 
                        8: [  
                                (8, 20, 90000),
                                (21, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        12: [  
                                (12, 19, 90000),
                                (20, float('inf'), 120000)
                            
                            ]       
            },
            "Gestor de ventas Nuevo":{ 
                        1: [  
                                (1, 12, 23000),
                                (13, 19, 90000),
                                (20, float('inf'), 120000)
                            
                            ]       
            }},
        "MEDELLIN": {
            "Gestor de ventas Senior":{ 
                        8: [  
                                (8, 20, 90000),
                                (21, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        12: [  
                                (12, 19, 90000),
                                (20, float('inf'), 120000)
                            
                            ]       
            }},
        "MONTERIA": {
            "Gestor de ventas Senior":{ 
                        8: [  
                                (8, 20, 90000),
                                (21, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        12: [  
                                (12, 19, 90000),
                                (20, float('inf'), 120000)
                            
                            ]       
            }},
        "NEIVA": {
            "Gestor de ventas Senior":{ 
                        8: [  
                                (8, 20, 90000),
                                (21, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        12: [  
                                (12, 19, 90000),
                                (20, float('inf'), 120000)
                            
                            ]       
            },
            "Gestor de ventas Nuevo":{ 
                        1: [  
                                (1, 3, 23000),
                                (3, 5, 90000),
                                (6, float('inf'), 120000)
                            
                            ]       
            }},
        "SANTA MARTA": {
            "Gestor de ventas Senior":{ 
                        8: [  
                                (8, 20, 90000),
                                (21, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        12: [  
                                (12, 19, 90000),
                                (20, float('inf'), 120000)
                            
                            ]       
            }},
        "SINCELEJO": {
            "Gestor de ventas Senior":{ 
                        8: [  
                                (8, 20, 90000),
                                (21, float('inf'), 120000)
                            ]},
            "Gestor de ventas Junior":{ 
                        12: [  
                                (12, 19, 90000),
                                (20, float('inf'), 120000)
                            
                            ]       
            }},               
        "CALI": {
            "Gestor de ventas Senior":{ 
                        13: [  
                                (13, 19, 100000),
                                (20, float('inf'), 130000)
                            ]},
            "Gestor de ventas Junior":{ 
                        15: [  
                                (15, 19, 100000),
                                (20, float('inf'), 130000)
                            ]          
            }}, 
        "PEREIRA": {
            "Gestor de ventas Senior":{ 
                        13: [  
                                (13, 19, 100000),
                                (20, float('inf'), 130000)
                            ]},
            "Gestor de ventas Junior":{ 
                        15: [  
                                (15, 19, 100000),
                                (20, float('inf'), 130000)
                            
                            ],
                        13: [  
                                (13, 19, 100000),
                                (20, float('inf'), 130000)
                            
                            ]     
                            }},
        "BUCARAMANGA": {
            "Gestor de ventas Senior":{ 
                        13: [  
                                (13, 19, 100000),
                                (20, float('inf'), 130000)
                            ]},
            "Gestor de ventas Junior":{ 
                        13: [  
                                (13, 19, 100000),
                                (20, float('inf'), 130000)
                            
                            ],
                        15: [  
                                (15, 19, 100000),
                                (20, float('inf'), 130000)
                            
                            ]     }},
         "TUNJA": {
            "Gestor de ventas Senior":{ 
                        11: [  
                                (11, 19, 100000),
                                (20, float('inf'), 130000)
                            
                            ]  },
            "Gestor de ventas Junior":{ 
                        15: [  
                                (15, 19, 100000),
                                (20, float('inf'), 130000)
                            
                            ]       
            }}            
    }}

In [406]:
def asignar_comision_especializaciones(lider, sede, cargo, meta, ventas):

    # 1️⃣ Validar líder
    if lider not in esquema_comisiones_especializaciones:
        return 0

    # 2️⃣ Validar sede
    if sede not in esquema_comisiones_especializaciones[lider]:
        return 0

    # 3️⃣ Validar cargo
    if cargo not in esquema_comisiones_especializaciones[lider][sede]:
        return 0

    # 4️⃣ Validar meta
    if meta not in esquema_comisiones_especializaciones[lider][sede][cargo]:
        return 0

    # 5️⃣ Validar cumplimiento de meta
    if ventas < meta:
        return 0

    # 6️⃣ Obtener los pisos correspondientes a la meta
    rangos = esquema_comisiones_especializaciones[lider][sede][cargo][meta]

    # 7️⃣ Buscar el piso correspondiente según las ventas
    for min_rango, max_rango, comision in rangos:

        if min_rango <= ventas <= max_rango:

            # Comisión por venta × número de ventas
            return ventas * comision

    return 0

In [407]:
ventas_asesor_especializaciones["COMISION_TOTAL_ESPECIALIZACION"] = ventas_asesor_especializaciones.apply(
    lambda row: asignar_comision_especializaciones(
        row["LIDER"],
        row["SEDE"],
        row["CARGO"],
        row["Meta"],
        row["VENTAS_ESPECIALIZACION"]
    ),
    axis=1
)
ventas_asesor_especializaciones['COMISION_UNITARIA_ESPECIALIZACION'] = ventas_asesor_especializaciones['COMISION_TOTAL_ESPECIALIZACION'] / ventas_asesor_especializaciones['VENTAS_ESPECIALIZACION']
ventas_asesor_especializaciones['%CUMPLIMIENTO_ESPECIALIZACION'] = (
    (ventas_asesor_especializaciones['VENTAS_ESPECIALIZACION'] / ventas_asesor_especializaciones['Meta']) * 100).round(2).astype(str) + '%'
ventas_asesor_especializaciones

,Documento Ejecutivo,VENTAS_ESPECIALIZACION,GESTOR CORREO,CARGO,Meta,LIDER,SEDE,LOCALIDAD,COMISION_TOTAL_ESPECIALIZACION,COMISION_UNITARIA_ESPECIALIZACION,%CUMPLIMIENTO_ESPECIALIZACION
0,1000224303,1,GUISELLE_VALENZUELA@CUN.EDU.CO,Gestor de ventas Nuevo,1,SANDRA,IBAGUE,IBAGUE,23000,23000.0,100.0%
1,1005337691,2,KAREN_AGUILAR@CUN.EDU.CO,Gestor de ventas Junior,15,SANDRA,BUCARAMANGA,BUCARAMANGA,0,0.0,13.33%
2,1006053851,23,VALENTINA_MONTOYAC@CUN.EDU.CO,Gestor de ventas Junior,15,SANDRA,CALI,CALI,2990000,130000.0,153.33%
3,1023938622,1,LUIS_MUNOZMU@CUN.EDU.CO,Gestor de ventas Junior,15,SANDRA,PEREIRA,PEREIRA,0,0.0,6.67%
4,1025521082,18,DAVID_ORJUELAC@CUN.EDU.CO,Gestor de ventas Junior,15,SANDRA,TUNJA,TUNJA,1800000,100000.0,120.0%
5,1033377195,27,LAURA_BEDOYAC@CUN.EDU.CO,Gestor de ventas Junior,12,SANDRA,MEDELLIN,MEDELLIN,3240000,120000.0,225.0%
6,1049637832,17,NATHALIA_FUQUENEY@CUN.EDU.CO,Gestor de ventas Junior,15,SANDRA,TUNJA,TUNJA,1700000,100000.0,113.33%
7,1053616791,23,MIRYAM_CALLEJASP@CUN.EDU.CO,Gestor de ventas Junior,15,SANDRA,TUNJA,TUNJA,2990000,130000.0,153.33%
8,1053787428,1,DAVID_QUICENO@CUN.EDU.CO,Gestor de ventas Senior,8,SANDRA,MEDELLIN,MEDELLIN,0,0.0,12.5%
9,1067283629,17,VICTOR_TOBON@CUN.EDU.CO,Gestor de ventas Junior,12,SANDRA,MONTERIA,MONTERIA,1530000,90000.0,141.67%


SE ARMA EL DATAFRAME FINAL

In [ ]:
ventas_asesor_virtuales = ventas_asesor_virtuales.rename(columns={'Meta':'META_VIRTUAL'})
ventas_asesor_presenciales = ventas_asesor_presenciales.rename(columns={'Meta':'META_PRESENCIAL'})
ventas_asesor_especializaciones = ventas_asesor_especializaciones.rename(columns={'Meta':'META_ESPECIALIZACION'})
base_pago = ventas_asesor_virtuales.merge(
    ventas_asesor_presenciales[
        [
            'Documento Ejecutivo',
            'VENTAS_PRESENCIALES',
            'META_PRESENCIAL',
            'COMISION_TOTAL_PRESENCIAL',
            'COMISION_UNITARIA_PRESENCIAL',
            '%CUMPLIMIENTO_PRESENCIAL'
        ]
    ],
    how='left',
    on='Documento Ejecutivo'
).copy()


,Documento Ejecutivo,VENTAS_VIRTUALES,GESTOR CORREO,CARGO,META_VIRTUAL,LIDER,SEDE,LOCALIDAD,COMISION_TOTAL_VIRTUAL,COMISION_UNITARIA_VIRTUAL,%CUMPLIMIENTO_VIRTUAL,VENTAS_PRESENCIALES,META_PRESENCIAL,COMISION_TOTAL_PRESENCIAL,COMISION_UNITARIA_PRESENCIAL,%CUMPLIMIENTO_PRESENCIAL
0,1000224303,2,GUISELLE_VALENZUELA@CUN.EDU.CO,Gestor de ventas Nuevo,1.0,SANDRA,IBAGUE,IBAGUE,46000,23000.0,200.0%,NaN,NaN,NaN,NaN,NaN
1,1001344725,4,JULIANA_AUDOR@CUN.EDU.CO,Gestor de ventas Nuevo,1.0,SANDRA,NEIVA,NEIVA,92000,23000.0,400.0%,NaN,NaN,NaN,NaN,NaN
2,1004734611,3,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,nan%,NaN,NaN,NaN,NaN,NaN
3,1005337691,7,KAREN_AGUILAR@CUN.EDU.CO,Gestor de ventas Junior,21.0,SANDRA,BUCARAMANGA,BUCARAMANGA,0,0.0,33.33%,NaN,NaN,NaN,NaN,NaN
4,1006053851,1,VALENTINA_MONTOYAC@CUN.EDU.CO,Gestor de ventas Junior,21.0,SANDRA,CALI,CALI,0,0.0,4.76%,NaN,NaN,NaN,NaN,NaN
5,1023938622,6,LUIS_MUNOZMU@CUN.EDU.CO,Gestor de ventas Junior,21.0,SANDRA,PEREIRA,PEREIRA,0,0.0,28.57%,NaN,NaN,NaN,NaN,NaN
6,1025521082,4,DAVID_ORJUELAC@CUN.EDU.CO,Gestor de ventas Junior,21.0,SANDRA,TUNJA,TUNJA,0,0.0,19.05%,NaN,NaN,NaN,NaN,NaN
7,1033377195,1,LAURA_BEDOYAC@CUN.EDU.CO,Gestor de ventas Junior,17.0,SANDRA,MEDELLIN,MEDELLIN,0,0.0,5.88%,2.0,17.0,0.0,0.0,11.76%
8,1036672351,17,JUAN_BRANPE@CUN.EDU.CO,Gestor de ventas Junior,17.0,SANDRA,MEDELLIN,MEDELLIN,1530000,90000.0,100.0%,NaN,NaN,NaN,NaN,NaN
9,1053616791,3,MIRYAM_CALLEJASP@CUN.EDU.CO,Gestor de ventas Junior,21.0,SANDRA,TUNJA,TUNJA,0,0.0,14.29%,NaN,NaN,NaN,NaN,NaN


In [436]:
base_pago_final = base_pago.merge(
    ventas_asesor_especializaciones[
        [
       'Documento Ejecutivo', 'VENTAS_ESPECIALIZACION','META_ESPECIALIZACION', 
       'COMISION_TOTAL_ESPECIALIZACION', 'COMISION_UNITARIA_ESPECIALIZACION',
       '%CUMPLIMIENTO_ESPECIALIZACION'
        ]
    ],
    how='left',
    on='Documento Ejecutivo'
).copy()

In [437]:
columnas_ventas = [
    'VENTAS_VIRTUALES',
    'VENTAS_PRESENCIALES',
    'VENTAS_ESPECIALIZACION'
]

columnas_comisiones = [
    'COMISION_TOTAL_VIRTUAL',
    'COMISION_TOTAL_PRESENCIAL',
    'COMISION_TOTAL_ESPECIALIZACION'
]

base_pago_final['VENTAS_TOTALES'] = (
    base_pago_final[columnas_ventas].fillna(0).sum(axis=1)
)

base_pago_final['COMISION_FINAL'] = (
    base_pago_final[columnas_comisiones].fillna(0).sum(axis=1)
)
base_pago_final['PERIODO_PAGO'] = periodo_pago
base_pago_final = base_pago_final[['Documento Ejecutivo','GESTOR CORREO', 'CARGO', 'LIDER', 'SEDE', 'LOCALIDAD', 'PERIODO_PAGO','VENTAS_VIRTUALES', 
       'META_VIRTUAL','%CUMPLIMIENTO_VIRTUAL', 'COMISION_UNITARIA_VIRTUAL','COMISION_TOTAL_VIRTUAL',
       'VENTAS_PRESENCIALES', 'META_PRESENCIAL','%CUMPLIMIENTO_PRESENCIAL','COMISION_UNITARIA_PRESENCIAL','COMISION_TOTAL_PRESENCIAL',
       'VENTAS_ESPECIALIZACION', 'META_ESPECIALIZACION', '%CUMPLIMIENTO_ESPECIALIZACION', 'COMISION_UNITARIA_ESPECIALIZACION',
       'COMISION_TOTAL_ESPECIALIZACION','VENTAS_TOTALES', 'COMISION_FINAL']]

In [438]:
base_pago_final

,Documento Ejecutivo,GESTOR CORREO,CARGO,LIDER,SEDE,LOCALIDAD,PERIODO_PAGO,VENTAS_VIRTUALES,META_VIRTUAL,%CUMPLIMIENTO_VIRTUAL,...,%CUMPLIMIENTO_PRESENCIAL,COMISION_UNITARIA_PRESENCIAL,COMISION_TOTAL_PRESENCIAL,VENTAS_ESPECIALIZACION,META_ESPECIALIZACION,%CUMPLIMIENTO_ESPECIALIZACION,COMISION_UNITARIA_ESPECIALIZACION,COMISION_TOTAL_ESPECIALIZACION,VENTAS_TOTALES,COMISION_FINAL
0,1000224303,GUISELLE_VALENZUELA@CUN.EDU.CO,Gestor de ventas Nuevo,SANDRA,IBAGUE,IBAGUE,202609,2,1.0,200.0%,...,NaN,NaN,NaN,1.0,1.0,100.0%,23000.0,23000.0,3.0,69000.0
1,1001344725,JULIANA_AUDOR@CUN.EDU.CO,Gestor de ventas Nuevo,SANDRA,NEIVA,NEIVA,202609,4,1.0,400.0%,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,92000.0
2,1004734611,NaN,NaN,NaN,NaN,NaN,202609,3,NaN,nan%,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,0.0
3,1005337691,KAREN_AGUILAR@CUN.EDU.CO,Gestor de ventas Junior,SANDRA,BUCARAMANGA,BUCARAMANGA,202609,7,21.0,33.33%,...,NaN,NaN,NaN,2.0,15.0,13.33%,0.0,0.0,9.0,0.0
4,1006053851,VALENTINA_MONTOYAC@CUN.EDU.CO,Gestor de ventas Junior,SANDRA,CALI,CALI,202609,1,21.0,4.76%,...,NaN,NaN,NaN,23.0,15.0,153.33%,130000.0,2990000.0,24.0,2990000.0
5,1023938622,LUIS_MUNOZMU@CUN.EDU.CO,Gestor de ventas Junior,SANDRA,PEREIRA,PEREIRA,202609,6,21.0,28.57%,...,NaN,NaN,NaN,1.0,15.0,6.67%,0.0,0.0,7.0,0.0
6,1025521082,DAVID_ORJUELAC@CUN.EDU.CO,Gestor de ventas Junior,SANDRA,TUNJA,TUNJA,202609,4,21.0,19.05%,...,NaN,NaN,NaN,18.0,15.0,120.0%,100000.0,1800000.0,22.0,1800000.0
7,1033377195,LAURA_BEDOYAC@CUN.EDU.CO,Gestor de ventas Junior,SANDRA,MEDELLIN,MEDELLIN,202609,1,17.0,5.88%,...,11.76%,0.0,0.0,27.0,12.0,225.0%,120000.0,3240000.0,30.0,3240000.0
8,1036672351,JUAN_BRANPE@CUN.EDU.CO,Gestor de ventas Junior,SANDRA,MEDELLIN,MEDELLIN,202609,17,17.0,100.0%,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.0,1530000.0
9,1053616791,MIRYAM_CALLEJASP@CUN.EDU.CO,Gestor de ventas Junior,SANDRA,TUNJA,TUNJA,202609,3,21.0,14.29%,...,NaN,NaN,NaN,23.0,15.0,153.33%,130000.0,2990000.0,26.0,2990000.0


In [445]:
no_validos_total = pd.concat([df_novalidos_virtual_final,df_novalidos_presencial_final,df_novalidos_especializacion_final])
no_validos_total['PERIODO_PAGO'] = periodo_pago
no_validos_total = no_validos_total[['DOC_ALUM', 'PERIODO', 'ESTADO_PAGO', 'FuerzaComercialFinal', 'NUEVO',
       'CLASE_ACTUAL', 'TIPO', 'PROGRAMA', 'EST_MATRICULADO',
       'Propietario del registro', 'Modalidad', 'Correo Ejecutivo','Documento Ejecutivo', 'Regional',
       'PERIODO_PAGO', 'MOTIVO']]
no_validos_total.shape

(160, 16)

In [446]:
validos_total = pd.concat([df_validos_virtuales_final,df_validos_presenciales_final,df_validos_especializaciones_final])
validos_total['PERIODO_PAGO'] = periodo_pago
validos_total = validos_total[['DOC_ALUM', 'PERIODO', 'ESTADO_PAGO', 'FuerzaComercialFinal', 'NUEVO',
       'CLASE_ACTUAL', 'TIPO', 'PROGRAMA', 'EST_MATRICULADO',
       'Propietario del registro', 'Modalidad', 'Correo Ejecutivo', 'Regional',
       'Documento Ejecutivo','Propietario_de_Oportunidad_Name', 'Fase', 'Hora_de_modificación', 'PERIODO_PAGO']]

In [450]:
with pd.ExcelWriter(
    f"LIQUIDACION_VINCULACIONES_REGIONALES_{periodo_pago}.xlsx",
    engine="openpyxl"
) as writer:

    base_pago_final.to_excel(
        writer,
        sheet_name="Comisiones_Asesores",
        index=False
    )

    validos_total.to_excel(
        writer,
        sheet_name="Validos",
        index=False
    )

    no_validos_total.to_excel(
        writer,
        sheet_name="No_Validas",
        index=False
    )

mal

In [ ]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 

        
        SELECT identificacion_btpr,concat(nombre_btpr,' ',segundo_nombre_btpr,' ',primer_apellido_btpr,' ',
                        segundo_apellido_btpr) as Nombre
        FROM HUB_persona
 ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_persona = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

DBAPIError: (pyodbc.Error) ('01000', '[01000] [Microsoft][ODBC SQL Server Driver][DBNETLIB]ConnectionWrite (send()). (10054) (SQLExecDirectW); [01000] [Microsoft][ODBC SQL Server Driver][DBNETLIB]Error general de red. Consulte la documentación de la red. (11)')
[SQL:  

        
        SELECT identificacion_btpr,concat(nombre_btpr,' ',segundo_nombre_btpr,' ',primer_apellido_btpr,' ',
                        segundo_apellido_btpr) as Nombre
        FROM HUB_persona
 ]
(Background on this error at: https://sqlalche.me/e/20/dbapi)

In [ ]:
df_persona=df_persona.drop_duplicates()
df_persona.shape

(665174, 2)

In [ ]:
df_final=pd.merge(df_piso,df_persona[['Nombre','identificacion_btpr']],how='left',left_on='Documento Ejecutivo',
                  right_on='identificacion_btpr')
df_final=df_final.drop(columns='identificacion_btpr')
df_final

,Documento_Estudiante,PERIODO_y,Nombre_de_Oportunidad,Propietario_de_Oportunidad_Name,Hora_de_creación,Fecha_de_cierre,Propietario_de_Oportunidad,Fase,Hora_de_modificación,Programa_de_interés,...,EST_MATRICULADO,_merge,VENTAS_ASESOR,Refe,Ventas Totales,Meta,LIDER,SEDE,CARGO,Nombre


In [ ]:
##df_final.loc[(df_final['Documento Ejecutivo'] == '1143244777'),'Meta']=1
##df_final.loc[(df_final['Documento Ejecutivo'] == '1081796706'),'Meta']=1
##df_final.loc[(df_final['Documento Ejecutivo'] == '44156610'),'Meta']=1
##df_final.loc[(df_final['Documento Ejecutivo'] == '1110553023'),'Meta']=1
##df_final.loc[(df_final['Documento Ejecutivo'] == '1112496766'),'Meta']=1


In [ ]:
### Definir los rangos de comisión
##rangos = [
##    (1,15,23000),
##    (15,20,80000), 
##    (20,30,90000),
##    (31, float('inf'), 110000)
##]
###rangos = [
###    (1,15,23000), 
###    (15,20,80000), #ELIMINAR DESPUÉS
###    (21, float('inf'), 120000)
###]
##
### Función para asignar la comisión según el número de pagos
##def asignar_comision(piso,pagos):
##    if piso == 0:
##        return 0
##    else:
##        if pagos < piso:
##            return 0
##        else:
##            for min_rango, max_rango, comision in rangos:
##                if min_rango <= pagos <= max_rango:
##                    return comision
##    return 0  # Si no entra en ningún rango (caso raro)
##
### Aplicar la función al DataFrame
##df_final["Comision Unitaria"] = df_final.apply(lambda row: asignar_comision(row['Meta'], row['Ventas Totales']), axis=1)


In [ ]:
esquema_comisiones = {

    "SANDRA": {
        "26V03": {
            "Gestor de ventas Senior":{ 
                        33: [  
                                (33, 39, 70000),
                                (40, float('inf'), 100000)
                            ]},
            "Gestor de ventas Junior":{ 
                        18: [  
                                (18, 24, 70000),
                                (25, float('inf'), 100000)
                            
                            ]       
            },

            "Gestor Grandes Cuentas":{ 
                        60: [  
                                (60, 79, 80000),
                                (80, float('inf'), 100000)
                            ]}},

        "26ES3": {

           "Gestor de ventas Senior":{ 
                        20: [  
                                (20, 29, 70000),
                                (30, float('inf'), 100000)
                            ]},
            "Gestor de ventas Junior":{ 
                        10: [  
                                (10, 14, 70000),
                                (15, float('inf'), 100000)
                            ]       
            },

            "Gestor Grandes Cuentas":{ 
                        20: [  
                                (20, 29, 65000),
                                (30, float('inf'), 80000)
                            ]}},
        "26T03": {

            "Gestor de ventas Senior":{ 
                        1: [  
                               (1, float('inf'), 70000)
                            ]},
            "Gestor de ventas Junior":{ 
                        1: [  
                                (1, float('inf'), 70000)
                            
                            ]       
            },

            "Gestor Grandes Cuentas":{ 
                        1: [   
                                (1, float('inf'), 70000)
                            ]}}
    }}

In [ ]:
def asignar_comision(lider, periodo, cargo, meta,ventas):

    # 1️⃣ Validar líder
    if lider not in esquema_comisiones:
        return 0

    # 2️⃣ Validar periodo
    if periodo not in esquema_comisiones[lider]:
        return 0

    # 3️⃣ Validar sede
    if cargo not in esquema_comisiones[lider][periodo]:
        return 0
    # 5️⃣ Validar cumplimiento de meta
    if ventas < meta:
        return 0

    rangos = esquema_comisiones[lider][periodo][cargo][meta]

    # 4️⃣ Evaluar rangos según ventas
    for min_rango, max_rango, comision in rangos:
        if min_rango <= ventas <= max_rango:
            return comision

    return 0


In [ ]:
df_final["Comision Unitaria"] = df_final.apply(
    lambda row: asignar_comision(
        row["LIDER "],
        row["PERIODO_y"],
        row["CARGO"],
        row["Meta"],
        row["Ventas Totales"]
    ),
    axis=1
)



In [ ]:
df_final.loc[df_final['ESTADO_PAGO']=='NO PAGO','Comision Unitaria'] = 0

In [ ]:
#Calcular la comision total
df_final['Comision Total'] = df_final['Ventas Totales']*df_final['Comision Unitaria']

In [ ]:
df_final.columns

Index(['Documento_Estudiante', 'PERIODO_y', 'Nombre_de_Oportunidad',
       'Propietario_de_Oportunidad_Name', 'Hora_de_creación',
       'Fecha_de_cierre', 'Propietario_de_Oportunidad', 'Fase',
       'Hora_de_modificación', 'Programa_de_interés', 'PERIODO_x', 'Id',
       'Nombre_de_Interesado', 'Creado_por', 'Hora_de_la_última_actividad',
       'Correo_electrónico', 'llave1', 'Documento Ejecutivo', 'ESTADO_PAGO',
       'FuerzaComercialFinal', 'FEC_PAGO_LIQ', 'NUEVO', 'CLASE_ACTUAL', 'TIPO',
       'PROGRAMA', 'EST_MATRICULADO', '_merge', 'VENTAS_ASESOR', 'Refe',
       'Ventas Totales', 'Meta', 'LIDER ', 'SEDE', 'CARGO', 'Nombre',
       'Comision Unitaria', 'Comision Total'],
      dtype='object')

In [ ]:
df_asesores = df_final[['Nombre','Documento Ejecutivo','Ventas Totales', 'Meta',
            'Comision Unitaria', 'Comision Total' ]]
df_asesores.drop_duplicates(subset='Nombre')
df_asesores

,Nombre,Documento Ejecutivo,Ventas Totales,Meta,Comision Unitaria,Comision Total


In [ ]:
df_asesores_2 = df_final.groupby('Documento Ejecutivo', as_index=False).agg({
    'Nombre': 'first',               # Primer nombre encontrado por documento
    'Ventas Totales': 'first',         # Suma total de ventas
    'Meta': 'mean',                   # Suma total de metas
    'Comision Unitaria' : 'first',
    'Comision Total' : 'first'
})


In [ ]:
df_asesores_2

,Documento Ejecutivo,Nombre,Ventas Totales,Meta,Comision Unitaria,Comision Total


In [ ]:
df_final

,Documento_Estudiante,PERIODO_y,Nombre_de_Oportunidad,Propietario_de_Oportunidad_Name,Hora_de_creación,Fecha_de_cierre,Propietario_de_Oportunidad,Fase,Hora_de_modificación,Programa_de_interés,...,VENTAS_ASESOR,Refe,Ventas Totales,Meta,LIDER,SEDE,CARGO,Nombre,Comision Unitaria,Comision Total


In [ ]:
df_novalidos

,Hora agregado,Numero de recibo,Propietario del registro,Nombre de Estudiante,Número de Documento de estudiante,Tipo de inscripción,Programa vendido,Modalidad,Periodo,Fecha de pago,...,Regional,Documento Ejecutivo_x,Número de aprobación de pago,Tipo de Financiación,Convenio,Valor pagado,Documento Ejecutivo_y,Documento Ejecutivo,Llave,MOTIVO


In [ ]:
#finalisimo

In [ ]:
with pd.ExcelWriter(
    "LIQUIDACION_VINCULACIONES_ABRIL_MAYO_26T03_REGIONALES.xlsx",
    engine="openpyxl"
) as writer:

    df_final.to_excel(
        writer,
        sheet_name="Validos",
        index=False
    )

    df_asesores_2.to_excel(
        writer,
        sheet_name="Comisiones_Asesores",
        index=False
    )

    df_novalidos.to_excel(
        writer,
        sheet_name="No_Validas",
        index=False
    )


In [ ]:
RR

NameError: name 'RR' is not defined

In [ ]:
df_final.to_excel("Liquidacion_vinculaciones_NOVIEMBRE_POS.xlsx", index=False)
df_asesores_2.to_excel("Liquidacion_comisiones_asesores_NOVIEMBRE_POS.xlsx", index=False)
df_novalidos.to_excel("Liquidacion_vinculaciones_No_validos_NOVIEMBRE_POS.xlsx", index=False)

In [ ]:
df_final['PERIODO_FINANCIERO'] = '202510'
df_final['EstadoFinal'] = 'VALIDO'
df_final['Reint_Stv'] = 0
df_final = df_final.drop(columns=['CLASE_ACTUAL','NUEVO'])
df_final = df_final.rename(columns={'PERIODO_y':'PERIODO','PROGRAMA':'NOM_PROGRAMA'})
df_final

,Documento_Estudiante,PERIODO,Nombre_de_Oportunidad,Propietario_de_Oportunidad_Name,Hora_de_creación,Fecha_de_cierre,Propietario_de_Oportunidad,Fase,Hora_de_modificación,Programa_de_interés,...,VENTAS_ASESOR,Refe,Ventas Totales,Meta,Nombre,Comision Unitaria,Comision Total,PERIODO_FINANCIERO,EstadoFinal,Reint_Stv
0,1000694004,2026A,8989-1000694004-250904090704,Vinculaciones Jeffer Sebastian Vargas Jimenez,04/09/2025 09:07,02/10/2025,4402173000785669009,Matriculado,16/10/2025 17:32,ADMINISTRACION DE EMPRESAS,...,2,0,2,40,JEFFER SEBASTIAN VARGAS JIMENEZ,0,0,202510,VALIDO,0
1,1000973546,2025D,8883-1000973546-250821064812,DANIEL FELIPE PRIETO ALVARADO,21/08/2025 18:48,21/08/2025,4402173000669519001,Matriculado,26/09/2025 12:26,INGENIERIA DE SISTEMAS,...,4,0,4,1,DANIEL FELIPE PRIETO ALVARADO,23000,92000,202510,VALIDO,0
2,1000987583,2026A,8973-1000987583-250812095636,paula andrea vence madariaga,12/08/2025 09:56,15/09/2025,4402173000808268013,Matriculado,26/09/2025 12:56,ADMINISTRACION EN SERVICIOS DE SALUD,...,2,0,2,1,PAULA ANDREA VENCE MADARIAGA,23000,46000,202510,VALIDO,0
3,1004462297,2026A,8997-1004462297-250827025501,Vinculaciones Rafael Pacheco,27/08/2025 14:55,30/08/2025,4402173000926976001,Matriculado,26/09/2025 10:12,DISEÑO GRAFICO,...,2,0,2,1,RAFAEL ALFONSO PACHECO ESCOBAR,23000,46000,202510,VALIDO,0
4,1010106273,2025D,8973-1010106273-250829050526,DANIEL FELIPE PRIETO ALVARADO,29/08/2025 17:05,08/09/2025,4402173000669519001,Matriculado,26/09/2025 10:04,ADMINISTRACION EN SERVICIOS DE SALUD,...,4,0,4,1,DANIEL FELIPE PRIETO ALVARADO,23000,92000,202510,VALIDO,0
5,1014478970,2025D,9048-1014478970-250902012226,paula andrea vence madariaga,06/09/2025 15:02,02/09/2025,4402173000808268013,Matriculado,29/09/2025 10:46,INGENIERIA DE SISTEMAS,...,2,0,2,1,PAULA ANDREA VENCE MADARIAGA,23000,46000,202510,VALIDO,0
6,1019120883,2026A,8989-1019120883-250902015322,Vinculaciones Jeffer Sebastian Vargas Jimenez,02/09/2025 13:53,01/10/2025,4402173000785669009,Matriculado,16/10/2025 17:27,ADMINISTRACION DE EMPRESAS,...,2,0,2,40,JEFFER SEBASTIAN VARGAS JIMENEZ,0,0,202510,VALIDO,0
7,1022325122,2025D,9048-1022325122-250922102049,DANIEL FELIPE PRIETO ALVARADO,22/09/2025 22:20,22/09/2025,4402173000669519001,Matriculado,29/09/2025 10:22,INGENIERIA DE SISTEMAS,...,4,0,4,1,DANIEL FELIPE PRIETO ALVARADO,23000,92000,202510,VALIDO,0
8,1030587209,25V05,8989-1030587209-250827094251,Vinculaciones Lina Paola Hernandez Palacios,27/08/2025 21:42,17/09/2025,4402173000781008015,Matriculado,26/09/2025 10:04,ADMINISTRACION DE EMPRESAS,...,1,0,1,40,LINA PAOLA HERNANDEZ PALACIOS,0,0,202510,VALIDO,0
9,1034657459,2026A,8997-1034657459-250811043747,Vinculaciones Pedro Nel Rangel Luna,11/08/2025 16:37,14/08/2025,4402173000785669007,Matriculado,26/09/2025 12:56,DISEÑO GRAFICO,...,1,0,1,40,PEDRO NEL RANGEL LUNA,0,0,202510,VALIDO,0


In [ ]:
df_final = df_final[['Documento_Estudiante', 'PERIODO', 'Nombre_de_Oportunidad',
       'Propietario_de_Oportunidad_Name', 'Hora_de_creación',
       'Fecha_de_cierre', 'Propietario_de_Oportunidad', 'Fase',
       'Hora_de_modificación', 'Programa_de_interés', 'Id',
       'Nombre_de_Interesado', 'Creado_por', 'Hora_de_la_última_actividad',
       'Correo_electrónico', 'llave1', 'Documento Ejecutivo', 'ESTADO_PAGO',
       'FuerzaComercialFinal', 'FEC_PAGO_LIQ', 'TIPO', 'NOM_PROGRAMA',
       '_merge', 'Reint_Stv', 'EstadoFinal', 'VENTAS_ASESOR', 'Refe',
       'Ventas Totales', 'Meta', 'Nombre', 'Comision Unitaria',
       'Comision Total', 'PERIODO_FINANCIERO']]

In [ ]:
# Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'COMISIONES_VINCULACIONES_ACTUALES'
schema_name = 'Comisiones'

# Send the data to sql server
df_final.to_sql(
    name=table_name,
    schema=schema_name,
    con=engine,
    if_exists='append',
    index=False,
    chunksize = 10000
)

# End connection
engine.dispose()

In [ ]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 

        
        select * from comisiones.COMISIONES_VINCULACIONES_ACTUALES
 ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_historico = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [ ]:
df_historico.columns

Index(['Documento_Estudiante', 'PERIODO', 'Nombre_de_Oportunidad',
       'Propietario_de_Oportunidad_Name', 'Hora_de_creación',
       'Fecha_de_cierre', 'Propietario_de_Oportunidad', 'Fase',
       'Hora_de_modificación', 'Programa_de_interés', 'Id',
       'Nombre_de_Interesado', 'Creado_por', 'Hora_de_la_última_actividad',
       'Correo_electrónico', 'llave1', 'Documento Ejecutivo', 'ESTADO_PAGO',
       'FuerzaComercialFinal', 'FEC_PAGO_LIQ', 'TIPO', 'NOM_PROGRAMA',
       '_merge', 'Reint_Stv', 'EstadoFinal', 'VENTAS_ASESOR', 'Refe',
       'Ventas Totales', 'Meta', 'Nombre', 'Comision Unitaria',
       'Comision Total', 'PERIODO_FINANCIERO'],
      dtype='object')

In [ ]:
df = pd.read_excel('Liquidacion_vinculaciones_No_validos_AGOSTO_SEPTIEMBRE_v6.xlsx', sheet_name='Hoja2')

In [ ]:
para_orden = df['documento'].unique()
valores_unicos_df1_orden = "', '".join(map(str, para_orden))
valores_unicos_df1_orden = f"'{valores_unicos_df1_orden}'"


In [ ]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(f''' 
        SELECT distinct DOC_ALUM as Documento_Estudiante_zoho,PERIODO,ESTADO_PAGO,fuerzazoho as FuerzaComercialFinal, FEC_PAGO_LIQ, NUEVO, CLASE_ACTUAL, TIPO, PROGRAMA, est_matriculado, documento
                          FROM ZOHO.Base_Personas WITH(NOLOCK)
        WHERE  doc_alum in ({valores_unicos_df1_orden})
 ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_pos = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()
df_pos.shape

(69, 11)

In [ ]:
df['documento'] = df['documento'].astype(str)

In [ ]:
df

,documento,periodo
0,1019070872,26ET1
1,1073810835,25v05
2,1090412616,26ET1
3,10305558829,26es1
4,1082897374,26es2
5,1031146614,26es1
6,1005319598,25v05
7,1064727721,2025D
8,6000769,25v05
9,1022351813,26ES1


In [ ]:
df_pos_2 = df.merge(df_pos, how='inner', left_on='documento', right_on='Documento_Estudiante_zoho')
df_pos_2.drop_duplicates(inplace=True)
df_pos_2.shape

(69, 13)

In [ ]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(f''' 
        Select distinct 
o.Nombre_de_Oportunidad,
o.Propietario_de_Oportunidad_Name,
o.Hora_de_creación,
o.Fecha_de_cierre,
o.Propietario_de_Oportunidad,
o.Fase,
o.Hora_de_modificación,
o.Número_de_Documento,
o.Programa_de_interés,
o.Periodo,
o.Id,
o.Nombre_de_Interesado,
o.Creado_por,
o.Hora_de_la_última_actividad,
u.Correo_electrónico
from zoho.CRM.Oportunidades o
left join zoho.CRM.Usuarios U
ON o.Propietario_de_Oportunidad = u.Id
where Número_de_Documento in ({valores_unicos_df1_orden})

 ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_2 = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()
df_2.shape

(64, 15)

In [ ]:
df_2

,Nombre_de_Oportunidad,Propietario_de_Oportunidad_Name,Hora_de_creación,Fecha_de_cierre,Propietario_de_Oportunidad,Fase,Hora_de_modificación,Número_de_Documento,Programa_de_interés,Periodo,Id,Nombre_de_Interesado,Creado_por,Hora_de_la_última_actividad,Correo_electrónico
0,4852-1024575919-211111123659,Vinculaciones Regionales Cesar Parra,11/11/2021 12:36,None,4402173000085759532,Matriculado,29/11/2021 16:11,1024575919,BOGOTA CENTRO / INGENIERIA DE SISTEMAS / TECNI...,None,4402173000121730001,4402173000121702138,4402173000000251001,29/11/2021 16:11,cesar_parra@cun.edu.co
1,5147-92541481-211109123512,victor_pedraza@cun.edu.co victor_pedraza@cun.e...,09/11/2021 12:35,None,4402173000003605037,PreInscrito,19/09/2024 17:49,92541481,BOGOTA CENTRO / CONTADURIA PUBLICA / TECNICO /...,21V06,4402173000121099001,4402173000121092039,4402173000000251001,19/09/2024 17:49,asesorcundigital18@cun.edu.co
2,7026-1006117688-240620020938,Contact Luisa Maria Velandia Gomez,20/06/2024 14:09,20/06/2024,4402173000011756279,Matriculado,20/06/2024 14:13,1006117688,ESPECIALIZACION EN PAZ Y DESARROLLO TERRITORIAL,24ET3,4402173000510587027,4402173000275344403,4402173000000251001,20/06/2024 14:13,asesorcundigital67@cun.edu.co
3,7900-1070387768-240924010927,Vinculaciones NATALIA REMOLINA HERRERA,24/09/2024 13:09,27/09/2024,4402173000751702001,Matriculado,11/08/2025 11:50,1070387768,DISEO DE MODAS,2025A,4402173000553652936,4402173000551893207,4402173000000251001,11/08/2025 11:50,natalia_remolinah@cun.edu.co
4,7959-1014290554-241128053143,Contact Daniel Camilo Pinzon Pachon,28/11/2024 17:31,28/11/2024,4402173000583230011,Matriculado,29/11/2024 10:35,1014290554,DISEÑO GRAFICO,24V06,4402173000593702164,4402173000593734175,4402173000000251001,29/11/2024 10:35,daniel_pinzonp@cun.edu.co
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,9101-1031147855-250930112959,Vinculaciones NATALIA REMOLINA HERRERA,30/09/2025 11:29,01/10/2025,4402173000751702001,Matriculado,01/10/2025 23:14,1031147855,ESPECIALIZACION EN GESTION DE TECNOLOGIAS DE L...,25ET4,4402173000959465939,4402173000959465933,4402173000751702001,01/10/2025 23:14,natalia_remolinah@cun.edu.co
60,9101-80724804-250922043549,Vinculaciones NATALIA REMOLINA HERRERA,22/09/2025 16:35,23/09/2025,4402173000751702001,Matriculado,29/09/2025 12:17,80724804,ESPECIALIZACION EN GESTION DE TECNOLOGIAS DE L...,25ET4,4402173000946459974,4402173000946459968,4402173000751702001,29/09/2025 12:17,natalia_remolinah@cun.edu.co
61,9250-1012368710-251017020826,Vinculaciones NATALIA REMOLINA HERRERA,17/10/2025 14:08,None,4402173000751702001,Interes basico,17/10/2025 14:10,1012368710,INGENIERIA INDUSTRIAL,25V06,4402173000989454137,4402173000989454131,4402173000751702001,17/10/2025 14:10,natalia_remolinah@cun.edu.co
62,ESP48-1006117688-230118033310,Contact paola_pena@cun.edu.co,18/01/2023 15:33,None,4402173000003607001,Matriculado,21/01/2023 09:57,1006117688,ESPECIALIZACION EN ANALITICA DE DATOS,23ET1,4402173000275436012,4402173000275344403,4402173000000251001,14/06/2024 16:45,asesorcundigital7@cun.edu.co


In [ ]:
df_3 = df.merge(df_2, how='inner', left_on='documento', right_on='Número_de_Documento')
df_3.drop_duplicates(inplace=True)
df_3

,documento,periodo,Nombre_de_Oportunidad,Propietario_de_Oportunidad_Name,Hora_de_creación,Fecha_de_cierre,Propietario_de_Oportunidad,Fase,Hora_de_modificación,Número_de_Documento,Programa_de_interés,Periodo,Id,Nombre_de_Interesado,Creado_por,Hora_de_la_última_actividad,Correo_electrónico
0,1019070872,26ET1,8945-1019070872-250912103758,Vinculaciones NATALIA REMOLINA HERRERA,12/09/2025 10:37,19/09/2025,4402173000751702001,Matriculado,01/10/2025 08:56,1019070872,ESPECIALIZACIÓN EN DESARROLLO ORGANIZACIONAL Y...,26ES1,4402173000925966386,4402173000925966380,4402173000751702001,01/10/2025 08:56,natalia_remolinah@cun.edu.co
1,1073810835,25v05,8872-1073810835-250915025710,Vinculaciones NATALIA REMOLINA HERRERA,15/09/2025 14:57,None,4402173000751702001,Interes basico,29/09/2025 10:43,1073810835,INGENIERIA INDUSTRIAL,25V05,4402173000933224823,4402173000933224817,4402173000751702001,29/09/2025 10:43,natalia_remolinah@cun.edu.co
2,1090412616,26ET1,8945-1090412616-250912121952,Vinculaciones NATALIA REMOLINA HERRERA,12/09/2025 12:19,15/09/2025,4402173000751702001,Matriculado,01/10/2025 08:56,1090412616,ESPECIALIZACIÓN EN DESARROLLO ORGANIZACIONAL Y...,None,4402173000925966986,4402173000925966980,4402173000751702001,01/10/2025 08:56,natalia_remolinah@cun.edu.co
3,1082897374,26es2,8953-1082897374-250912060847,Vinculaciones NATALIA REMOLINA HERRERA,12/09/2025 18:08,15/09/2025,4402173000751702001,Matriculado,29/09/2025 12:17,1082897374,ESPECIALIZACIÓN EN ALTA GERENCIA,26ES1,4402173000927051145,4402173000927051139,4402173000751702001,29/09/2025 12:17,natalia_remolinah@cun.edu.co
4,1031146614,26es1,8949-1031146614-250913123747,Vinculaciones NATALIA REMOLINA HERRERA,13/09/2025 12:37,None,4402173000751702001,Interes real,29/09/2025 12:17,1031146614,ESPECIALIZACIÓN EN GERENCIA DE PROYECTOS,None,4402173000927203969,4402173000927203963,4402173000751702001,29/09/2025 12:17,natalia_remolinah@cun.edu.co
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,1121826358,26es1,8942-1121826358-251001050630,Vinculaciones NATALIA REMOLINA HERRERA,01/10/2025 17:06,01/10/2025,4402173000751702001,Matriculado,06/10/2025 16:01,1121826358,ESPECIALIZACIÓN EN PAZ Y DESARROLLO TERRITORIAL,26ES1,4402173000962333258,4402173000962333252,4402173000751702001,06/10/2025 16:01,natalia_remolinah@cun.edu.co
60,1014227769,26es1,8944-1014227769-251001060133,Vinculaciones NATALIA REMOLINA HERRERA,01/10/2025 18:01,01/10/2025,4402173000751702001,Matriculado,06/10/2025 16:01,1014227769,ESPECIALIZACIÓN EN GERENCIA EDUCATIVA,26ES1,4402173000962362953,4402173000962362947,4402173000751702001,06/10/2025 16:01,natalia_remolinah@cun.edu.co
61,1120579442,26es1,8951-1120579442-251001061012,Vinculaciones NATALIA REMOLINA HERRERA,01/10/2025 18:10,01/10/2025,4402173000751702001,Matriculado,06/10/2025 16:00,1120579442,ESPECIALIZACIÓN EN CONTRATACIÓN ESTATAL,26ES1,4402173000962406178,4402173000962406172,4402173000751702001,06/10/2025 16:00,natalia_remolinah@cun.edu.co
62,1070387768,2025D,7900-1070387768-240924010927,Vinculaciones NATALIA REMOLINA HERRERA,24/09/2024 13:09,27/09/2024,4402173000751702001,Matriculado,11/08/2025 11:50,1070387768,DISEO DE MODAS,2025A,4402173000553652936,4402173000551893207,4402173000000251001,11/08/2025 11:50,natalia_remolinah@cun.edu.co


In [ ]:
df_pos_2.to_excel('cruce_base_personas.xlsx',index=False)

In [ ]:
df_3.to_excel('cruce_zoho.xlsx',index=False)

In [ ]:
df = pd.read_excel(r'C:\Users\alejandro_arboledas\OneDrive - Corporación Unificada Nacional de Educación Superior - CUN\Documentos\PYTHON\VINCULACIONES\MODELO3\Liquidacion_comisiones_vinculaciones_AGOSTO_SEPTIEMBRE_v2.xlsx', sheet_name='No validos')

In [ ]:
# Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'VINCULACIONES_NO_VALIDOS'
schema_name = 'Comisiones'

# Send the data to sql server
df.to_sql(
    name=table_name,
    schema=schema_name,
    con=engine,
    if_exists='replace',
    index=False,
    chunksize = 10000
)

# End connection
engine.dispose()

In [ ]:
df_2 = pd.read_excel(r'C:\Users\alejandro_arboledas\OneDrive - Corporación Unificada Nacional de Educación Superior - CUN\Documentos\PYTHON\VINCULACIONES\MODELO3\Liquidacion_comisiones_asesores_AGOSTO_SEPTIEMBRE_v6.xlsx', sheet_name='Hoja2')
df_2

,Documento Ejecutivo,Nombre,Ventas Totales,Meta,Comision Unitaria,Comision Total,PERIODO_FINANCIERO
0,1026592868,NATALIA REMOLINA HERRERA,132,40,97000,12804000.0,202509
1,1018465686,ANGEL ANTONIO RIVERA PADUA,66,1,69000,4554000.0,202509
2,1128053909,MIGUEL ANGEL PALOMINO PERALTA,19,40,0,0.0,202509
3,1002152146,HUGO LEONARDO ARISTIZABAL MARTELO,13,40,0,0.0,202509
4,1110464615,CESAR AUGUSTO CANDIA MENDOZA,3,1,23000,69000.0,202509
5,1012388460,DAIRON FELIPE TELLEZ ZAMBRANO,42,40,57500,2415000.0,202509
6,1015393814,CAROLINA ROCIO VILORIA PEDREROS,39,40,0,0.0,202509
7,1033679718,LUZ DARY RAMIREZ QUEVEDO,65,40,69000,4485000.0,202509
8,1102850551,MARIA ANDREA UPARELA OVIEDO,13,40,0,0.0,202509
9,1235041897,STEFANY OSPINA TARRA,2,1,23000,46000.0,202509


In [ ]:
# Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'COMISIONES_ASESOR_VINCULACIONES_ACTUALES'
schema_name = 'Comisiones'

# Send the data to sql server
df_2.to_sql(
    name=table_name,
    schema=schema_name,
    con=engine,
    if_exists='append',
    index=False,
    chunksize = 10000
)

# End connection
engine.dispose()